# MineToday V131 — Global Champion-Aligned BM25 Rank-2 Micro-Extension

**Frozen confirmatory experiment contract.** V131 starts from the exact public-scoring V130 candidate (`28dd2121…`, public NDCG@5 = 0.66173), freezes all 70 existing Top1 edits, reconstructs V129/V130 analogue authority fold-safely, and tests exactly one new residual mechanism: a globally pooled exact-ΔNDCG Ridge head with the five preregistered V128 BM25 streams. The primary policy may add at most five rank-2-only Top1 score-value swaps. No pseudo-GT, no new encoder, no pair one-hot, no K/IDCG feature, no coverage/alpha search, and no lower-order action family.

The notebook intentionally emits both a diagnostic candidate and a fail-closed primary file. The decision report is authoritative; filenames alone are not.


In [1]:

# =========================
# 0. Runtime / dependencies / frozen V131 configuration
# =========================
import os, sys, re, io, gc, json, ast, html, math, time, base64, gzip, hashlib, random, warnings, unicodedata, subprocess, platform
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, Counter

os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("OMP_NUM_THREADS","4")
os.environ.setdefault("OPENBLAS_NUM_THREADS","4")
os.environ.setdefault("MKL_NUM_THREADS","4")
os.environ.setdefault("NUMEXPR_NUM_THREADS","4")
os.environ.setdefault("PYTHONHASHSEED","20260907")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY","1")
warnings.filterwarnings("ignore")

def _ensure_runtime():
    import importlib.metadata as im
    need=[]
    try: im.version("lightgbm")
    except Exception: need.append("lightgbm==4.6.0")
    try:
        tv=im.version("transformers")
        if int(tv.split('.')[0])>=5: need += ["transformers==4.57.1","sentence-transformers==5.1.1"]
    except Exception: need += ["transformers==4.57.1","sentence-transformers==5.1.1"]
    try: im.version("sentence-transformers")
    except Exception:
        if not any(x.startswith("sentence-transformers") for x in need): need.append("sentence-transformers==5.1.1")
    try: im.version("einops")
    except Exception: need.append("einops>=0.8.0")
    if need:
        print("Installing compatibility dependencies before model imports:",need)
        subprocess.check_call([sys.executable,"-m","pip","install","-q",*need])
_ensure_runtime()

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse import hstack, csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
from IPython.display import display

SEED=20260907
random.seed(SEED); np.random.seed(SEED)

@dataclass(frozen=True)
class CFG:
    seed:int=SEED
    n_outer:int=5
    n_inner:int=3
    word_max_features:int=60000
    char_max_features:int=80000
    near_threshold:float=0.90
    lgb_estimators:int=160
    lgb_lr:float=0.03
    lgb_num_leaves:int=15
    lgb_max_depth:int=4
    lgb_min_child:int=80
    lgb_subsample:float=0.80
    lgb_colsample:float=0.75
    lgb_reg_alpha:float=0.5
    lgb_reg_lambda:float=8.0
    lgb_seeds:tuple=(20260907,20260917,20260927)
    action_alpha:float=10.0
    v129_coverage:float=0.05
    v130_coverage:float=0.02
    v131_coverage:float=0.005
    submit_gain:float=0.00015
    borderline_gain:float=0.00005
    catastrophic_stress:float=-0.00050
    bootstraps:int=2000
    smoke:bool=bool(int(os.getenv("V131_SMOKE","0")))
    run_stress:bool=bool(int(os.getenv("V131_RUN_STRESS","1")))

cfg=CFG()
if cfg.smoke:
    cfg=CFG(n_outer=2,n_inner=2,word_max_features=8000,char_max_features=12000,lgb_estimators=30,
            lgb_seeds=(20260907,),bootstraps=100,smoke=True,run_stress=False)
OUT=Path("/kaggle/working/v131") if Path("/kaggle/working").exists() else Path("./v131")
OUT.mkdir(parents=True,exist_ok=True)
print("CFG",asdict(cfg))
print("numpy",np.__version__,"pandas",pd.__version__,"scipy",scipy.__version__,"sklearn",__import__('sklearn').__version__,"lightgbm",lgb.__version__)
try: print(subprocess.check_output(["nvidia-smi","-L"],text=True).strip())
except Exception: print("GPU not visible; full V131 targets Kaggle T4x2.")


Installing compatibility dependencies before model imports: ['transformers==4.57.1', 'sentence-transformers==5.1.1']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.7 MB/s eta 0:00:00
CFG {'seed': 20260907, 'n_outer': 5, 'n_inner': 3, 'word_max_features': 60000, 'char_max_features': 80000, 'near_threshold': 0.9, 'lgb_estimators': 160, 'lgb_lr': 0.03, 'lgb_num_leaves': 15, 'lgb_max_depth': 4, 'lgb_min_child': 80, 'lgb_subsample': 0.8, 'lgb_colsample': 0.75, 'lgb_reg_alpha': 0.5, 'lgb_reg_lambda': 8.0, 'lgb_seeds': (20260907, 20260917, 20260927), 'action_alpha': 10.0, 'v129_coverage': 0.05, 'v130_coverage': 0.02, 'v131_coverage': 0.005, 'submit_gain': 0.00015, 'borderline_gain': 5e-05, 'catastrophic_stress': -0.0005, 'bootstraps': 2000, '

In [2]:
# =========================
# 1. Exact historical champion + V129 provenance + exact public-positive V130 base + organizer data
# =========================
CHAMPION_SHA256="443785fb534df9820ce57bfe450a2768881625bf7ae37aa849dd0d9f2afc957d"
CHAMPION_TOP5_SHA256="7c0d29a7309b7283a043b7d37ca5556ba7c0ca2270b0137a6e1e0878a69a5fe7"
CHAMPION_GZ_B64="""..."""
NON_S0_TEST_USERS=set(["U_4005", "U_4015", "U_4017", "U_4043", "U_4044", "U_4058", "U_4068", "U_4069", "U_4070", "U_4075", "U_4087", "U_4089", "U_4091", "U_4098", "U_4105", "U_4118", "U_4121", "U_4122", "U_4123", "U_4130", "U_4131", "U_4137", "U_4138", "U_4169", "U_4178", "U_4183", "U_4186", "U_4203", "U_4214", "U_4219", "U_4232", "U_4233", "U_4237", "U_4265", "U_4266", "U_4269", "U_4274", "U_4285", "U_4289", "U_4300", "U_4304", "U_4320", "U_4335", "U_4345", "U_4354", "U_4363", "U_4379", "U_4389", "U_4394", "U_4402", "U_4407", "U_4430", "U_4436", "U_4438", "U_4442", "U_4480", "U_4515", "U_4527", "U_4532", "U_4542", "U_4545", "U_4555", "U_4569", "U_4580", "U_4582", "U_4596", "U_4601", "U_4633", "U_4639", "U_4650", "U_4661", "U_4663", "U_4665", "U_4672", "U_4675", "U_4687", "U_4717", "U_4718", "U_4725", "U_4730", "U_4751", "U_4759", "U_4763", "U_4827", "U_4828", "U_4838", "U_4886", "U_4887", "U_4893", "U_4896", "U_4897", "U_4905", "U_4919", "U_4933", "U_4942", "U_4951", "U_4956", "U_4960", "U_4965", "U_4971", "U_4978", "U_4988", "U_4989"])
V25_PROTECTED_TEST_USERS={"U_4047","U_4953","U_4074","U_4176","U_4491","U_4567","U_4836","U_4929"}

CHAMPION_BYTES=gzip.decompress(base64.b64decode(CHAMPION_GZ_B64.encode()))
assert hashlib.sha256(CHAMPION_BYTES).hexdigest()==CHAMPION_SHA256
champion=pd.read_csv(io.BytesIO(CHAMPION_BYTES))


V129_BASE_SHA256="34f334429c2113ba5b96283a7974fad472068ea08047ba3275fc44057e3a95df"
V129_PUBLIC_LB=0.66075
V129_BASE_GZ_B64="""..."""
FROZEN_V129_TRAIN_SELECTED=set(['U_0002', 'U_0011', 'U_0032', 'U_0036', 'U_0062', 'U_0073', 'U_0119', 'U_0129', 'U_0130', 'U_0140', 'U_0206', 'U_0232', 'U_0258', 'U_0321', 'U_0337', 'U_0342', 'U_0399', 'U_0403', 'U_0404', 'U_0439', 'U_0444', 'U_0448', 'U_0455', 'U_0468', 'U_0488', 'U_0490', 'U_0511', 'U_0515', 'U_0526', 'U_0541', 'U_0549', 'U_0562', 'U_0598', 'U_0609', 'U_0618', 'U_0625', 'U_0648', 'U_0655', 'U_0662', 'U_0687', 'U_0690', 'U_0707', 'U_0722', 'U_0723', 'U_0728', 'U_0739', 'U_0793', 'U_0834', 'U_0858', 'U_0930', 'U_0931', 'U_0943', 'U_0947', 'U_0953', 'U_0956', 'U_0973', 'U_0997', 'U_1007', 'U_1037', 'U_1038', 'U_1045', 'U_1053', 'U_1058', 'U_1071', 'U_1084', 'U_1133', 'U_1145', 'U_1146', 'U_1204', 'U_1211', 'U_1214', 'U_1224', 'U_1243', 'U_1275', 'U_1278', 'U_1305', 'U_1312', 'U_1373', 'U_1428', 'U_1454', 'U_1491', 'U_1548', 'U_1570', 'U_1575', 'U_1584', 'U_1586', 'U_1594', 'U_1595', 'U_1676', 'U_1687', 'U_1754', 'U_1763', 'U_1844', 'U_1872', 'U_1963', 'U_1966', 'U_1968', 'U_1986', 'U_2002', 'U_2039', 'U_2051', 'U_2054', 'U_2066', 'U_2082', 'U_2102', 'U_2136', 'U_2142', 'U_2161', 'U_2183', 'U_2207', 'U_2262', 'U_2281', 'U_2284', 'U_2314', 'U_2352', 'U_2374', 'U_2379', 'U_2407', 'U_2409', 'U_2414', 'U_2420', 'U_2434', 'U_2474', 'U_2493', 'U_2503', 'U_2510', 'U_2533', 'U_2541', 'U_2554', 'U_2570', 'U_2586', 'U_2600', 'U_2601', 'U_2605', 'U_2620', 'U_2662', 'U_2703', 'U_2704', 'U_2710', 'U_2721', 'U_2739', 'U_2778', 'U_2781', 'U_2785', 'U_2799', 'U_2835', 'U_2868', 'U_2886', 'U_2924', 'U_2991', 'U_3023', 'U_3132', 'U_3133', 'U_3177', 'U_3199', 'U_3216', 'U_3217', 'U_3262', 'U_3269', 'U_3312', 'U_3326', 'U_3338', 'U_3370', 'U_3380', 'U_3415', 'U_3488', 'U_3509', 'U_3511', 'U_3521', 'U_3541', 'U_3551', 'U_3606', 'U_3607', 'U_3612', 'U_3613', 'U_3625', 'U_3656', 'U_3671', 'U_3690', 'U_3693', 'U_3710', 'U_3711', 'U_3749', 'U_3752', 'U_3762', 'U_3769', 'U_3801', 'U_3831', 'U_3868', 'U_3872', 'U_3877', 'U_3886', 'U_3893', 'U_3907', 'U_3914', 'U_3942', 'U_3944', 'U_3982', 'U_3986', 'U_3994'])
FROZEN_V129_TEST_SELECTED=set(['U_4027', 'U_4028', 'U_4062', 'U_4071', 'U_4128', 'U_4151', 'U_4153', 'U_4155', 'U_4213', 'U_4224', 'U_4246', 'U_4281', 'U_4309', 'U_4329', 'U_4331', 'U_4356', 'U_4397', 'U_4412', 'U_4417', 'U_4462', 'U_4478', 'U_4490', 'U_4501', 'U_4524', 'U_4549', 'U_4617', 'U_4619', 'U_4626', 'U_4642', 'U_4656', 'U_4664', 'U_4703', 'U_4713', 'U_4715', 'U_4720', 'U_4731', 'U_4742', 'U_4780', 'U_4783', 'U_4788', 'U_4799', 'U_4805', 'U_4807', 'U_4840', 'U_4843', 'U_4844', 'U_4898', 'U_4936', 'U_4979', 'U_4985'])
assert len(FROZEN_V129_TRAIN_SELECTED)==200 and len(FROZEN_V129_TEST_SELECTED)==50
V129_BASE_BYTES=gzip.decompress(base64.b64decode(V129_BASE_GZ_B64.encode()))
assert hashlib.sha256(V129_BASE_BYTES).hexdigest()==V129_BASE_SHA256

REQUIRED={"chat_history.csv","modules_catalog.csv","user_assessments.csv","train_relevance.csv","test.csv","sample_submission.csv"}
def locate_data():
    roots=[Path("/kaggle/input"),Path("."),Path("/mnt/data")]
    for root in roots:
        if not root.exists(): continue
        cands=[root]
        if root.name in {"input","data"} or root==Path("/kaggle/input"):
            cands += [p for p in root.rglob("*") if p.is_dir()]
        for d in cands:
            try: names={p.name for p in d.iterdir() if p.is_file()}
            except Exception: continue
            if REQUIRED.issubset(names): return d
    raise FileNotFoundError("MineToday organizer CSV directory not found under /kaggle/input.")

DATA=locate_data(); print("DATA",DATA)
chat=pd.read_csv(DATA/"chat_history.csv")
modules=pd.read_csv(DATA/"modules_catalog.csv")
assess=pd.read_csv(DATA/"user_assessments.csv")
train=pd.read_csv(DATA/"train_relevance.csv")
test=pd.read_csv(DATA/"test.csv")
sample=pd.read_csv(DATA/"sample_submission.csv")
MODULES=list(train.columns[1:]); M=len(MODULES); NTR=len(train); NTE=len(test); N=NTR+NTE
assert M==17 and NTR==4000 and NTE==1000
assert MODULES==list(sample.columns[1:])==list(champion.columns[1:])
assert champion.user_id.astype(str).tolist()==test.user_id.astype(str).tolist()
ALL_USERS=train.user_id.astype(str).tolist()+test.user_id.astype(str).tolist(); UID2I={u:i for i,u in enumerate(ALL_USERS)}
y=train[MODULES].to_numpy(np.float32)

# Frozen deployment geometry from direct V30/V31 ledgers: union=103 non-S0; exact S0 count=897.
assert len(NON_S0_TEST_USERS)==103
assert len(V25_PROTECTED_TEST_USERS)==8
TEST_ELIGIBLE=np.array([(u not in NON_S0_TEST_USERS) and (u not in V25_PROTECTED_TEST_USERS) for u in test.user_id.astype(str)],bool)
assert int(TEST_ELIGIBLE.sum())==889

DISCOUNT=1/np.log2(np.arange(2,7,dtype=np.float64))
def top5_idx(scores):
    s=np.asarray(scores); ix=np.argpartition(-s,4,axis=1)[:,:5]; loc=np.take_along_axis(s,ix,1)
    return np.take_along_axis(ix,np.argsort(-loc,axis=1,kind="stable"),1)
def top5_hash(scores): return hashlib.sha256(top5_idx(scores).astype(np.int16).tobytes()).hexdigest()
def per_user_ndcg(yy,ss):
    ix=top5_idx(ss); dcg=(np.take_along_axis(yy,ix,1)*DISCOUNT).sum(1)
    ideal=np.sort(yy,axis=1)[:,-5:][:,::-1]; idcg=(ideal*DISCOUNT).sum(1)
    return dcg/np.maximum(idcg,1e-12)
def ndcg5(yy,ss): return float(per_user_ndcg(yy,ss).mean())
def swap_values(scores,row,a,b):
    out=scores.copy(); out[row,a],out[row,b]=out[row,b],out[row,a]; return out

CHAMPION_SCORES=champion[MODULES].to_numpy(np.float32)
assert top5_hash(CHAMPION_SCORES)==CHAMPION_TOP5_SHA256
print("Champion verified; eligible S0 after V25 protection:",TEST_ELIGIBLE.sum())

v129_base=pd.read_csv(io.BytesIO(V129_BASE_BYTES))
assert v129_base.columns.tolist()==champion.columns.tolist()
assert v129_base.user_id.astype(str).tolist()==test.user_id.astype(str).tolist()
V129_BASE_SCORES=v129_base[MODULES].to_numpy(np.float32)
assert np.isfinite(V129_BASE_SCORES).all()
_ch=top5_idx(CHAMPION_SCORES); _vb=top5_idx(V129_BASE_SCORES)
_vchg=np.any(_ch!=_vb,axis=1)
_vchg_ids=set(test.user_id.astype(str).to_numpy()[_vchg])
assert int(_vchg.sum())==50 and _vchg_ids==FROZEN_V129_TEST_SELECTED
assert not np.any([set(a)!=set(b) for a,b in zip(_ch,_vb)])
assert np.array_equal(_ch[:,4],_vb[:,4])
V129_TEST_SELECTED=np.asarray([u in FROZEN_V129_TEST_SELECTED for u in test.user_id.astype(str)],bool)
V129_TRAIN_SELECTED=np.asarray([u in FROZEN_V129_TRAIN_SELECTED for u in train.user_id.astype(str)],bool)
V130_TEST_ELIGIBLE=TEST_ELIGIBLE & (~V129_TEST_SELECTED)
assert int(V130_TEST_ELIGIBLE.sum())==839
print("V129 base verified:",V129_BASE_SHA256,"public",V129_PUBLIC_LB,"frozen actions",int(V129_TEST_SELECTED.sum()),"residual eligible",int(V130_TEST_ELIGIBLE.sum()))


# --- Exact public-scoring V130 package: this, not submission_v130_primary.csv, is the protected V131 base. ---
V130_BASE_SHA256="28dd2121c72e910be8a01b31cd43d022c7fdf080d8e628bd8f30abc3a24bc93e"
V130_PUBLIC_LB=0.66173
V130_BASE_GZ_B64="""..."""
V130_BASE_BYTES=gzip.decompress(base64.b64decode(V130_BASE_GZ_B64.encode()))
assert hashlib.sha256(V130_BASE_BYTES).hexdigest()==V130_BASE_SHA256
v130_base=pd.read_csv(io.BytesIO(V130_BASE_BYTES))
assert v130_base.columns.tolist()==champion.columns.tolist()
assert v130_base.user_id.astype(str).tolist()==test.user_id.astype(str).tolist()
V130_BASE_SCORES=v130_base[MODULES].to_numpy(np.float32)
assert np.isfinite(V130_BASE_SCORES).all()
_o0=top5_idx(CHAMPION_SCORES); _o129=top5_idx(V129_BASE_SCORES); _o130=top5_idx(V130_BASE_SCORES)
V130_TEST_SELECTED=(_o0[:,0]!=_o130[:,0])
V130_INCREMENTAL_TEST_SELECTED=(_o129[:,0]!=_o130[:,0])
assert int(V130_TEST_SELECTED.sum())==70 and int(V130_INCREMENTAL_TEST_SELECTED.sum())==20
assert not np.any([set(a)!=set(b) for a,b in zip(_o0,_o130)])
assert np.array_equal(_o0[:,4],_o130[:,4])
assert np.all(V130_TEST_SELECTED<=TEST_ELIGIBLE), 'public-positive edits must lie inside protected S0 eligibility'
V131_TEST_ELIGIBLE=TEST_ELIGIBLE & (~V130_TEST_SELECTED)
assert int(V131_TEST_ELIGIBLE.sum())==819
print("V130 protected base verified:",V130_BASE_SHA256,"public",V130_PUBLIC_LB,"frozen Top1 edits",int(V130_TEST_SELECTED.sum()),"residual eligible",int(V131_TEST_ELIGIBLE.sum()))


DATA /kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026
Champion verified; eligible S0 after V25 protection: 889
V129 base verified: 34f334429c2113ba5b96283a7974fad472068ea08047ba3275fc44057e3a95df public 0.66075 frozen actions 50 residual eligible 839
V130 protected base verified: 28dd2121c72e910be8a01b31cd43d022c7fdf080d8e628bd8f30abc3a24bc93e public 0.66173 frozen Top1 edits 70 residual eligible 819


In [3]:

# =========================
# 2. Frozen V42 champion-aligned policy-anchor engine
# =========================
# This block is intentionally isolated conceptually from V129's retrieval anchor.
# It reconstructs the V30/V31 + B34/B45 policy geometry inside arbitrary train/validation splits.
all_users=ALL_USERS; uid2idx=UID2I; assessments=assess
N_MODULES=M; N_TRAIN=NTR; N_TEST=NTE
RIDGE_ALPHA=3.0; REFERENCE_W=0.45
ANCHOR_INNER_FOLDS=2 if cfg.smoke else 4
INTENT_INNER_FOLDS=2 if cfg.smoke else 4
LGB_ESTIMATORS=40 if cfg.smoke else 1200
LGB_EARLY_STOP=5 if cfg.smoke else 80
RANKER_ESTIMATORS=40 if cfg.smoke else 360
RANKER_LEARNING_RATE=0.04 if cfg.smoke else 0.025
ALPHA=0.16
INTENT_PARENT_STRENGTH=40.0; INTENT_CHILD_STRENGTH=30.0
BOOTSTRAPS=100 if cfg.smoke else 1200
EXPECTED_BOUNDARY_COUNTS={"B23":9,"B34":10,"B45":8}

def _seed_from_text(s): return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8],16)%(2**31-1)
top5_indices=top5_idx
Y_BIN=(y>0).astype(np.float32)
Y_TOP=np.zeros_like(y,np.float32); Y_TOP[np.arange(NTR),np.argmax(y,axis=1)]=1.0

# Robust assessment parsing — exact V3.1 mechanism
WORD_NUM = {"nol":0, "satu":1, "dua":2, "tiga":3, "empat":4, "lima":5}
SKILL_NAMES = ["python","sql","statistics","eda","ml","ml_eval","dl","genai","business","research"]

A = np.zeros((len(all_users), 10), np.float32)
A_INVALID = np.zeros_like(A)

for row in assessments.itertuples(index=False):
    try:
        d = json.loads(row.assessment_result)
    except Exception:
        d = ast.literal_eval(row.assessment_result)

    vals, invalid = [], []
    for v in d.values():
        sv = str(v).strip().lower()
        if sv in WORD_NUM:
            value, bad = float(WORD_NUM[sv]), 0
        else:
            try:
                value = float(v)
            except Exception:
                value = np.nan
            bad = int((not np.isfinite(value)) or value < 0 or value > 5)
            if bad:
                value = np.nan
        vals.append(value)
        invalid.append(bad)

    vals = np.asarray(vals, float)
    med = float(np.nanmedian(vals)) if np.isfinite(vals).any() else 2.0
    vals = np.where(np.isfinite(vals), vals, med)
    i = uid2idx[row.user_id]
    A[i] = vals
    A_INVALID[i] = invalid

print("invalid assessment cells:", int(A_INVALID.sum()), "/", A_INVALID.size)

# Exact V3.1 dataset-derived topic/style parser
TOPIC_PATTERNS = {
    "excel": r"\b(?:ms\.?\s*excel|excel|pivot(?: table)?|vlookup|hlookup|spreadsheet)\b",
    "python": r"\b(?:python|pyton|pithon|pandas(?:\s*&\s*| dan |\s+)?numpy|pandas|numpy)\b",
    "sql": r"\b(?:sql|escuel|query database|join tabel|database relasional|window function)\b",
    "statistics": r"\b(?:statistik(?:\s*&\s*probabilitas)?|probabilitas|a\s*/?\s*b testing|uji hipotesis|distribusi data)\b",
    "eda": r"\b(?:eda|exploratory data analysis|data cleaning|missing values?|matplotlib|seaborn)\b",
    "ml_eval": r"\b(?:evaluasi model(?: ml)?|metrik model|metric model|evaluasi performa model)\b",
    "mlops": r"\b(?:mlops|deployment model|deploy(?:ment)? model|ci\s*/?\s*cd|monitoring model|serving model|fastapi)\b",
    "cv": r"\b(?:computer vision|cnn|deep learning|deteksi objek|klasifikasi gambar|image classification)\b",
    "nlp": r"\b(?:natural language processing|nlp|analisis sentimen|sentiment analysis|word2vec|tf\s*-?\s*idf|transformer)\b",
    "genai": r"\b(?:generative ai|genai|large language model|llm|rag|chatgpt|chatbot|fine[- ]?tuning model bahasa)\b",
    "ml": r"\b(?:machine learning|bikin model ml|model machine learning|regresi\s*(?:&|dan)\s*klasifikasi|clustering|supervised|unsupervised|hyperparameter)\b",
    "business": r"\b(?:translate masalah bisnis ke data|menerjemahkan masalah bisnis|business(?:-| )to(?:-| )ds|permasalahan bisnis)\b",
    "dashboard": r"\b(?:dashboard|looker studio|tableau|visualisasi interaktif|storytelling data)\b",
    "webscrape": r"\b(?:web scrap(?:e|ing)|scraping|beautifulsoup|selenium|ekstraksi data.*website)\b",
    "git": r"\b(?:version control|git(?:hub|lab)?|repository|repo kode)\b",
    "prompt": r"\b(?:prompt engineering|prompting|merancang prompt|optimasi prompt)\b",
    "automation": r"\b(?:ai automation|otomasi|otomatis|automation|workflow|n8n)\b",
    "nocode": r"\b(?:no[ -]?code|zero coding|tanpa perlu (?:belajar )?(?:coding|koding)|gausah (?:coding|koding)|gak perlu (?:coding|koding)|tanpa (?:coding|koding))\b",
    "career": r"\b(?:portofolio|portfolio|cv|test interview|interview|karir|career|prospek kerja|negosiasi)\b",
}
TOPIC_RX = {k: re.compile(v, re.I) for k,v in TOPIC_PATTERNS.items()}
TOPICS = list(TOPIC_PATTERNS)

NOISE_RX = re.compile(
    r"reset password|email verifikasi|sertifikat kelulusan|pembayaran|cicilan|invoice|"
    r"akun.*(?:error|masalah)|lms.*(?:error|masalah)", re.I
)
STYLE_PATTERNS = {
    "beginner": r"pemula|dari nol|gaptek|belum pernah (?:ngoding|coding|koding)|background(?:nya)? bukan (?:it|data)|banting setir",
    "advanced": r"udah lumayan paham|sudah lumayan paham|stuck|prasyarat|syarat|\breq\b|basic apa|lanjutin",
    "urgent": r"urgent|secepatnya|sefastnya|intensif|seminggu|minggu depan",
    "budget": r"budget|diskon|worth it|cekak|fomo|kompetitor",
    "online": r"online|durasi|kapan buka|mulai minggu|jadwal",
    "nocode_style": r"tanpa perlu.*(?:coding|koding)|gausah.*(?:coding|koding)|gak perlu.*(?:coding|koding)|otomatis.*ai|solusi otomatis",
    "career_style": r"portofolio|portfolio|test interview|prospek kerja|karir|career",
}
STYLE_RX = {k: re.compile(v, re.I) for k,v in STYLE_PATTERNS.items()}
STYLES = list(STYLE_PATTERNS)

def clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"[^\w\s<>/&+-]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

def extract_topics(text):
    return [k for k,rx in TOPIC_RX.items() if rx.search(str(text))]

def make_skeleton(text):
    s = clean_text(text)
    for rx in TOPIC_RX.values():
        s = rx.sub(" <topic> ", s)
    s = re.sub(r"\b\d+\b", " <num> ", s)
    return re.sub(r"\s+", " ", s).strip()

def template_token(skeleton):
    return "tpl_" + hashlib.md5(skeleton.encode("utf-8")).hexdigest()[:10]

chat["timestamp"] = pd.to_datetime(chat["timestamp"], errors="coerce")
chat["clean"] = chat["user_chat_text"].fillna("").map(clean_text)
chat["is_noise"] = chat["user_chat_text"].fillna("").map(lambda x: bool(NOISE_RX.search(str(x))))
chat["topics"] = chat["user_chat_text"].fillna("").map(extract_topics)
chat["skeleton"] = chat["user_chat_text"].fillna("").map(make_skeleton)
for name,rx in STYLE_RX.items():
    chat["sty_"+name] = chat["user_chat_text"].fillna("").map(lambda x,r=rx: int(bool(r.search(str(x)))))

chat = chat.sort_values(["user_id","timestamp","chat_id"]).reset_index(drop=True)
user_chat = {u:g.copy() for u,g in chat.groupby("user_id", sort=False)}
print("administrative/noise rows:", int(chat["is_noise"].sum()), "/", len(chat))

# V3.1 template-aware sparse representation + structured user features
USER_FEATURES = np.zeros((len(all_users), 10 + 10 + 5 + len(STYLES)*2 + len(TOPICS)*5), np.float32)

def build_user(u):
    ui = uid2idx[u]
    g = user_chat.get(u)
    if g is None:
        learn = pd.DataFrame(columns=chat.columns)
        g = learn
    else:
        learn = g[~g["is_noise"]].copy()
        if len(learn) == 0:
            learn = g.copy()

    parts = []
    n = len(learn)
    for j,row in enumerate(learn.itertuples(index=False)):
        distance = n - 1 - j
        tag = "current" if distance == 0 else ("recent" if distance <= 2 else "older")
        parts.append(f"[{tag}] {row.clean}")
        parts.append(template_token(row.skeleton))
        for tp in row.topics:
            parts.append("topic_" + tp)
            if distance == 0:
                parts.extend(["latest_topic_" + tp] * 3)
            elif distance <= 2:
                parts.extend(["recent_topic_" + tp] * 2)
        for sty in STYLES:
            if getattr(row, "sty_"+sty):
                parts.append("style_" + sty)

    for j,skill in enumerate(SKILL_NAMES):
        v = int(round(float(A[ui,j])))
        band = "low" if v <= 1 else ("mid" if v <= 3 else "high")
        parts += [f"skill_{skill}_{v}", f"skill_{skill}_bin_{band}"]

    feats = list(A[ui]) + list(A_INVALID[ui])
    total = len(g)
    learning_count = len(learn)
    noise_count = int(g["is_noise"].sum()) if total else 0
    span = 0.0
    if total >= 2 and g["timestamp"].notna().sum() >= 2:
        span = float((g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 86400.0)
    feats += [total, learning_count, noise_count, span, min(total,6)]

    for sty in STYLES:
        feats.append(float(learn["sty_"+sty].sum()) if learning_count else 0.0)
    for sty in STYLES:
        feats.append(float(learn.iloc[-1]["sty_"+sty]) if learning_count else 0.0)

    rows = list(learn.itertuples(index=False))
    nfull = len(rows)
    for tp in TOPICS:
        count = rec_count = last_pos = 0.0
        latest = 0.0
        for j,row in enumerate(rows):
            if tp in row.topics:
                count += 1.0
                rec_count += float(np.exp(-0.7 * (nfull - 1 - j)))
                last_pos = max(last_pos, (j+1) / max(nfull,1))
        if rows and tp in rows[-1].topics:
            latest = 1.0
        feats += [count, rec_count, latest, last_pos, float(count > 0)]

    USER_FEATURES[ui] = np.asarray(feats, np.float32)
    return " ".join(parts)

texts = np.asarray([build_user(u) for u in all_users], dtype=object)
assert USER_FEATURES.shape == (5000, 134)
assert np.isfinite(USER_FEATURES).all()
print("USER_FEATURES:", USER_FEATURES.shape)
print("example augmented text:", texts[1][:500])


# V42 target-blind sparse text basis. Full 5,000-user vocabulary matches the frozen lineage.
word_vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.995,sublinear_tf=True,strip_accents="unicode",max_features=(8000 if cfg.smoke else 50000),dtype=np.float32)
char_vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True,max_features=(12000 if cfg.smoke else 60000),dtype=np.float32)
X_WORD=word_vec.fit_transform(texts); X_CHAR=char_vec.fit_transform(texts); X=hstack([X_WORD,X_CHAR],format="csr")
CONSENSUS3_OVERRIDES={"M_002":0.44,"M_006":0.55,"M_007":0.60,"M_010":0.40,"M_011":0.55,"M_012":0.55,"M_015":0.44,"M_017":0.53}
def consensus3_vector():
    w=np.full(N_MODULES,REFERENCE_W,np.float32)
    for mid,val in CONSENSUS3_OVERRIDES.items(): w[MODULES.index(mid)]=float(val)
    return w
CONS_W=consensus3_vector()

# Module-native lexical / intent / readiness surfaces inherited from V30 Base.
MODULE_TEXT = (
    modules.set_index("module_id").loc[MODULES, "module_name"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "description_and_syllabus"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "prerequisite_level"].astype(str)
).tolist()
MODULE_WORD = word_vec.transform(MODULE_TEXT)
MODULE_CHAR = char_vec.transform(MODULE_TEXT)
LEX_WORD = (X_WORD @ MODULE_WORD.T).toarray().astype(np.float32)
LEX_CHAR = (X_CHAR @ MODULE_CHAR.T).toarray().astype(np.float32)

MODULE_TOPIC_MAP = {
    "M_001":["excel"], "M_002":["python"], "M_003":["sql"],
    "M_004":["webscrape","python"], "M_005":["git"], "M_006":["statistics"],
    "M_007":["eda","python","statistics"], "M_008":["dashboard","sql","excel"],
    "M_009":["ml","ml_eval","eda","statistics"], "M_010":["cv","ml"],
    "M_011":["nlp","ml"], "M_012":["genai","nlp"], "M_013":["prompt","genai"],
    "M_014":["automation","genai"], "M_015":["nocode","genai"],
    "M_016":["mlops","ml","python","git"], "M_017":["career"],
}
MODULE_SKILL_MAP = {
    "M_001":[], "M_002":[0], "M_003":[1], "M_004":[0], "M_005":[9],
    "M_006":[2], "M_007":[0,2,3], "M_008":[1], "M_009":[2,3,4,5],
    "M_010":[4,6], "M_011":[4], "M_012":[4,7], "M_013":[7],
    "M_014":[7,8], "M_015":[7], "M_016":[0,4,5,9], "M_017":[8,9],
}

MODULE_COUNT = np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_RECENCY = np.zeros_like(MODULE_COUNT)
MODULE_LATEST = np.zeros_like(MODULE_COUNT)
LATEST_GROUP = np.empty(len(all_users),dtype=object); LATEST_GROUP[:] = "no_chat"
LATEST_INTENT_KEY = np.empty(len(all_users),dtype=object); LATEST_INTENT_KEY[:] = "intent:no_chat"

for u in all_users:
    ui=uid2idx[u]
    g=user_chat.get(u)
    if g is None or len(g)==0:
        continue
    learn=g[~g["is_noise"]].copy()
    if len(learn)==0:
        learn=g.copy()
    rows=list(learn.itertuples(index=False))
    if rows:
        LATEST_GROUP[ui]=rows[-1].skeleton or "empty"
        tps=sorted(set(rows[-1].topics))
        LATEST_INTENT_KEY[ui] = "intent:" + ("+".join(tps) if tps else "none")
    n=len(rows)
    for j,row in enumerate(rows):
        dist=n-1-j; w=float(np.exp(-0.7*dist)); row_topics=set(row.topics)
        for m,mid in enumerate(MODULES):
            hits=sum(tp in row_topics for tp in MODULE_TOPIC_MAP[mid])
            if hits:
                MODULE_COUNT[ui,m]+=hits
                MODULE_RECENCY[ui,m]+=w*hits
                if dist==0:
                    MODULE_LATEST[ui,m]=1.0

MODULE_READINESS=np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_INVALID=np.zeros_like(MODULE_READINESS)
for m,mid in enumerate(MODULES):
    idx=MODULE_SKILL_MAP[mid]
    if idx:
        MODULE_READINESS[:,m]=A[:,idx].mean(1)
        MODULE_INVALID[:,m]=A_INVALID[:,idx].mean(1)
    else:
        MODULE_READINESS[:,m]=A.mean(1)
        MODULE_INVALID[:,m]=A_INVALID.mean(1)

# Fixed readiness discretization consistent with the V3.1 low/mid/high assessment bins.
_ready_round=np.floor(MODULE_READINESS+0.5)
MODULE_READY_BAND=np.where(_ready_round<=1,0,np.where(_ready_round<=3,1,2)).astype(np.int8)

TRAIN_GROUP=np.asarray(LATEST_GROUP[:N_TRAIN],dtype=object)
TRAIN_INTENT=np.asarray(LATEST_INTENT_KEY[:N_TRAIN],dtype=object)
TEST_INTENT=np.asarray(LATEST_INTENT_KEY[N_TRAIN:],dtype=object)

prtxt=modules.set_index("module_id")["prerequisite_level"].astype(str).to_dict()
def module_level(mid):
    s=prtxt[mid].lower()
    if "pemula" in s: return 0
    if "menengah" in s: return 1
    if "lanjutan" in s: return 2
    if "ahli" in s: return 3
    return 4
MODULE_LEVEL=np.asarray([module_level(m) for m in MODULES],np.int16)
MODULE_LEVEL_ONEHOT=np.eye(5,dtype=np.float32)[MODULE_LEVEL]

# Ordinal LambdaRank labels: 0=irrelevant, 1=#6 ... 6=#1.
ORDINAL_GRADES=np.zeros_like(y,dtype=np.int16)
for i,row in enumerate(y):
    pos=np.flatnonzero(row>0)
    order=pos[np.argsort(-row[pos],kind="mergesort")]
    for rank,m in enumerate(order,1):
        ORDINAL_GRADES[i,m]=max(1,7-rank)
LABEL_GAIN=[0.0,0.25,0.40,0.55,0.70,0.85,1.00]

def user_strata_from_y(yy):
    yy=np.asarray(yy)
    k=(yy>0).sum(1).astype(int)
    top=np.argmax(yy,axis=1).astype(int)
    return (top*4+np.clip(k-3,0,3)).astype(int)

def make_group_folds_local(yy,groups,n_splits,seed):
    yy=np.asarray(yy); groups=np.asarray(groups,dtype=object)
    ns=min(int(n_splits),len(np.unique(groups)))
    if ns<2:
        sk=StratifiedKFold(n_splits=2,shuffle=True,random_state=int(seed))
        return list(sk.split(np.zeros(len(yy)),user_strata_from_y(yy)))
    sg=StratifiedGroupKFold(n_splits=ns,shuffle=True,random_state=int(seed))
    folds=list(sg.split(np.zeros(len(yy)),user_strata_from_y(yy),groups=groups))
    for tr,va in folds:
        assert len(set(groups[tr]).intersection(set(groups[va])))==0
    return folds

def make_group_folds_global(rows,n_splits,seed):
    rows=np.asarray(rows,int)
    local=make_group_folds_local(y[rows],TRAIN_GROUP[rows],n_splits,seed)
    return [(rows[tr],rows[va]) for tr,va in local]

print("catalog lexical matrices:",LEX_WORD.shape,LEX_CHAR.shape)
print("latest template groups:",len(np.unique(TRAIN_GROUP)))
print("latest intent signatures train/test:",len(np.unique(TRAIN_INTENT)),len(np.unique(TEST_INTENT)))
print("test intent signatures unseen in train:",len(set(TEST_INTENT)-set(TRAIN_INTENT)))
print("readiness band counts:",dict(zip(*np.unique(MODULE_READY_BAND[:N_TRAIN],return_counts=True))))
print("ordinal grade counts:",dict(zip(*np.unique(ORDINAL_GRADES,return_counts=True))))

ALL_INTENT=np.concatenate([np.asarray(TRAIN_INTENT,dtype=object),np.asarray(TEST_INTENT,dtype=object)])
# Strict fold-safe hierarchical Intent-State target encoding.
INTENTSTATE_KEYS = [
    "IS_PARENT_SEEN","IS_PARENT_SUPPORT_LOG","IS_PARENT_CONF",
    "IS_PARENT_POS","IS_PARENT_ORD","IS_PARENT_RAW",
    "IS_CHILD_SUPPORT_LOG","IS_CHILD_CONF",
    "IS_CHILD_POS","IS_CHILD_ORD","IS_CHILD_RAW","IS_CHILD_TOP5",
    "IS_POS_LOG_LIFT_GLOBAL","IS_ORD_DELTA_GLOBAL","IS_ORD_DELTA_PARENT","IS_RAW_DELTA_PARENT",
]

def _intent_targets(y_fit,grades_fit):
    y_fit=np.asarray(y_fit,np.float32); grades_fit=np.asarray(grades_fit,np.int16)
    return {
        "pos":(y_fit>0).astype(np.float32),
        "ord":(grades_fit.astype(np.float32)/6.0),
        "raw":y_fit.astype(np.float32),
        "top5":(grades_fit>=2).astype(np.float32),
    }

def fit_intentstate_bundle(y_fit,grades_fit,intent_fit,ready_band_fit):
    y_fit=np.asarray(y_fit,np.float32)
    grades_fit=np.asarray(grades_fit,np.int16)
    intent_fit=np.asarray(intent_fit,dtype=object)
    ready_band_fit=np.asarray(ready_band_fit,np.int8)
    n=len(y_fit)
    assert grades_fit.shape==y_fit.shape==ready_band_fit.shape
    T=_intent_targets(y_fit,grades_fit)
    glob={k:v.mean(0).astype(np.float32) for k,v in T.items()}

    parent={}
    child={}
    for key in np.unique(intent_fit):
        idx=np.flatnonzero(intent_fit==key)
        ns=int(len(idx))
        vals={}
        for name,M in T.items():
            sm=(M[idx].sum(0)+INTENT_PARENT_STRENGTH*glob[name])/(ns+INTENT_PARENT_STRENGTH)
            vals[name]=sm.astype(np.float32)
        parent[str(key)]={"n":ns,**vals}

        for m in range(N_MODULES):
            rb=ready_band_fit[idx,m]
            for b in (0,1,2):
                local=idx[rb==b]
                nc=int(len(local))
                if nc==0:
                    continue
                cv={}
                for name,M in T.items():
                    prior=float(vals[name][m])
                    cv[name]=float((M[local,m].sum()+INTENT_CHILD_STRENGTH*prior)/(nc+INTENT_CHILD_STRENGTH))
                child[(str(key),m,int(b))]={"n":nc,**cv}

    return {"n_fit":n,"global":glob,"parent":parent,"child":child}

def apply_intentstate_bundle(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object)
    ready_band=np.asarray(ready_band,np.int8)
    n=len(intent_keys)
    assert ready_band.shape==(n,N_MODULES)
    out={k:np.zeros((n,N_MODULES),np.float32) for k in INTENTSTATE_KEYS}
    glob=bundle["global"]
    logden=np.log1p(max(1,int(bundle["n_fit"])))

    for i,key0 in enumerate(intent_keys):
        key=str(key0)
        p=bundle["parent"].get(key)
        parent_seen=float(p is not None)
        if p is None:
            pn=0
            pvals={name:glob[name] for name in ("pos","ord","raw","top5")}
        else:
            pn=int(p["n"])
            pvals={name:p[name] for name in ("pos","ord","raw","top5")}
        pconf=pn/(pn+INTENT_PARENT_STRENGTH) if pn else 0.0
        pslog=np.log1p(pn)/logden if logden>0 else 0.0

        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            if c is None:
                cn=0
                cpos=float(pvals["pos"][m]); cord=float(pvals["ord"][m])
                craw=float(pvals["raw"][m]); ctop5=float(pvals["top5"][m])
            else:
                cn=int(c["n"]); cpos=float(c["pos"]); cord=float(c["ord"])
                craw=float(c["raw"]); ctop5=float(c["top5"])
            cconf=cn/(cn+INTENT_CHILD_STRENGTH) if cn else 0.0
            cslog=np.log1p(cn)/logden if logden>0 else 0.0
            gpos=float(glob["pos"][m]); gord=float(glob["ord"][m]); graw=float(glob["raw"][m])
            pord=float(pvals["ord"][m]); praw=float(pvals["raw"][m])

            out["IS_PARENT_SEEN"][i,m]=parent_seen
            out["IS_PARENT_SUPPORT_LOG"][i,m]=pslog
            out["IS_PARENT_CONF"][i,m]=pconf
            out["IS_PARENT_POS"][i,m]=float(pvals["pos"][m])
            out["IS_PARENT_ORD"][i,m]=pord
            out["IS_PARENT_RAW"][i,m]=praw
            out["IS_CHILD_SUPPORT_LOG"][i,m]=cslog
            out["IS_CHILD_CONF"][i,m]=cconf
            out["IS_CHILD_POS"][i,m]=cpos
            out["IS_CHILD_ORD"][i,m]=cord
            out["IS_CHILD_RAW"][i,m]=craw
            out["IS_CHILD_TOP5"][i,m]=ctop5
            out["IS_POS_LOG_LIFT_GLOBAL"][i,m]=float(np.clip(np.log((cpos+1e-4)/(gpos+1e-4)),-2.0,2.0))
            out["IS_ORD_DELTA_GLOBAL"][i,m]=cord-gord
            out["IS_ORD_DELTA_PARENT"][i,m]=cord-pord
            out["IS_RAW_DELTA_PARENT"][i,m]=craw-praw

    for k,v in out.items():
        if not np.isfinite(v).all():
            raise ValueError(f"non-finite Intent-State feature {k}")
    return out

def crossfit_intentstate_features(y_local,grades_local,intent_local,ready_band_local,groups_local,seed):
    y_local=np.asarray(y_local,np.float32)
    grades_local=np.asarray(grades_local,np.int16)
    intent_local=np.asarray(intent_local,dtype=object)
    ready_band_local=np.asarray(ready_band_local,np.int8)
    groups_local=np.asarray(groups_local,dtype=object)
    out={k:np.zeros_like(y_local,np.float32) for k in INTENTSTATE_KEYS}
    folds=make_group_folds_local(y_local,groups_local,INTENT_INNER_FOLDS,seed)
    seen=np.zeros(len(y_local),bool)
    for tr,va in folds:
        assert len(set(groups_local[tr]).intersection(set(groups_local[va])))==0
        bundle=fit_intentstate_bundle(y_local[tr],grades_local[tr],intent_local[tr],ready_band_local[tr])
        vv=apply_intentstate_bundle(bundle,intent_local[va],ready_band_local[va])
        for k in INTENTSTATE_KEYS:
            out[k][va]=vv[k]
        seen[va]=True
    assert seen.all(), "Intent-State crossfit did not cover every training row"
    return out

def intentstate_support_audit(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object); ready_band=np.asarray(ready_band,np.int8)
    supports=[]; parent_support=[]; parent_seen=[]
    for i,key0 in enumerate(intent_keys):
        key=str(key0); p=bundle["parent"].get(key)
        pn=int(p["n"]) if p is not None else 0
        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            supports.append(int(c["n"]) if c is not None else 0)
            parent_support.append(pn); parent_seen.append(int(p is not None))
    s=np.asarray(supports,int); ps=np.asarray(parent_support,int); pseen=np.asarray(parent_seen,int)
    rows=[]
    for thr in [1,5,10,20,30,50]:
        rows.append({"level":"child","threshold":thr,"coverage":float(np.mean(s>=thr))})
    rows.append({"level":"child","threshold":-1,"coverage":1.0,"median_support":float(np.median(s)),"p10_support":float(np.quantile(s,.10)),"p25_support":float(np.quantile(s,.25))})
    rows.append({"level":"parent","threshold":1,"coverage":float(np.mean(pseen>0)),"median_support":float(np.median(ps))})
    return pd.DataFrame(rows)

print("strict nested Intent-State TE helpers ready; features:",len(INTENTSTATE_KEYS))

def rank_matrix(scores):
    order=np.argsort(-np.asarray(scores),axis=1,kind="mergesort")
    ranks=np.empty_like(order,dtype=np.int16)
    ranks[np.arange(len(order))[:,None],order]=np.arange(1,N_MODULES+1,dtype=np.int16)[None,:]
    return ranks

def top5_order_change_rate(a,b):
    return float(np.mean(np.any(top5_indices(a)!=top5_indices(b),axis=1)))

def build_long_matrix_v31(rows_global,anchor,base,meta,intentstate=None):
    rows_global=np.asarray(rows_global,int); anchor=np.asarray(anchor,np.float32)
    base=np.asarray(base,np.float32); meta=np.asarray(meta,np.float32); n=len(rows_global)

    ranks=rank_matrix(anchor).astype(np.float32)
    top1=anchor.max(1,keepdims=True)
    sorted_scores=np.sort(anchor,axis=1)[:,::-1]
    cut5=sorted_scores[:,4:5]
    mu=anchor.mean(1,keepdims=True); sd=np.maximum(anchor.std(1,keepdims=True),1e-4)
    z=(anchor-mu)/sd
    order=np.argsort(-anchor,axis=1,kind="mergesort")
    prev_gap=np.zeros_like(anchor); next_gap=np.zeros_like(anchor)
    for i in range(n):
        oo=order[i]; ss=anchor[i,oo]
        pg=np.r_[0.0,ss[:-1]-ss[1:]]
        ng=np.r_[ss[:-1]-ss[1:],0.0]
        prev_gap[i,oo]=pg; next_gap[i,oo]=ng

    base_mats=[
        anchor, ranks/float(N_MODULES), (ranks<=5).astype(np.float32), top1-anchor, anchor-cut5,
        prev_gap,next_gap,z,base,meta,meta-base,
        LEX_WORD[rows_global],LEX_CHAR[rows_global],MODULE_COUNT[rows_global],MODULE_RECENCY[rows_global],
        MODULE_LATEST[rows_global],MODULE_READINESS[rows_global],MODULE_INVALID[rows_global],
    ]
    base_names=[
        "anchor_score","anchor_rank_norm","anchor_top5","top1_minus_score","score_minus_cut5",
        "prev_gap","next_gap","anchor_z","base_score","meta_score","meta_minus_base",
        "lex_word","lex_char","topic_count","topic_recency","topic_latest","readiness","invalid_readiness",
    ]
    pieces=[np.column_stack([m.reshape(-1) for m in base_mats]).astype(np.float32)]
    names=list(base_names)

    pieces.append(np.repeat(USER_FEATURES[rows_global],N_MODULES,axis=0).astype(np.float32))
    names += [f"user_f{i:03d}" for i in range(USER_FEATURES.shape[1])]
    pieces.append(np.tile(np.eye(N_MODULES,dtype=np.float32),(n,1)))
    names += [f"module_{m}" for m in MODULES]
    pieces.append(np.tile(MODULE_LEVEL_ONEHOT,(n,1)).astype(np.float32))
    names += [f"module_level_{i}" for i in range(5)]
    bounds={"base":sum(p.shape[1] for p in pieces)}

    if intentstate is not None:
        mats=[intentstate[k] for k in INTENTSTATE_KEYS]
        pieces.append(np.column_stack([m.reshape(-1) for m in mats]).astype(np.float32))
        names += [k.lower() for k in INTENTSTATE_KEYS]
        bounds["intentstate"]=sum(p.shape[1] for p in pieces)

    XX=np.hstack(pieces).astype(np.float32,copy=False)
    if XX.shape[0] != n*N_MODULES:
        raise AssertionError(XX.shape)
    if len(names)!=XX.shape[1]:
        raise AssertionError((len(names),XX.shape))
    if not np.isfinite(XX).all():
        raise ValueError("non-finite V31 long features")
    return XX,names,bounds

def make_ranker(seed):
    return lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=RANKER_ESTIMATORS,
        learning_rate=RANKER_LEARNING_RATE,
        num_leaves=15,
        min_child_samples=80,
        reg_alpha=1.0,
        reg_lambda=10.0,
        colsample_bytree=0.78,
        subsample=0.90,
        subsample_freq=1,
        max_bin=127,
        lambdarank_truncation_level=8,
        label_gain=LABEL_GAIN,
        random_state=int(seed),
        n_jobs=4,
        verbosity=-1,
        force_col_wise=True,
    )

def fit_ranker_predict(Xtr,Xva,grades_tr,n_users,seed):
    model=make_ranker(seed)
    group=np.full(int(n_users),N_MODULES,dtype=np.int32)
    model.fit(Xtr,grades_tr.reshape(-1).astype(np.int32),group=group)
    pred=model.predict(Xva).reshape(-1,N_MODULES).astype(np.float32)
    return model,pred

def align_ranker_to_anchor(raw_ranker,anchor):
    raw=np.asarray(raw_ranker,np.float32); anchor=np.asarray(anchor,np.float32)
    rmu=raw.mean(1,keepdims=True); rsd=np.maximum(raw.std(1,keepdims=True),1e-5)
    amu=anchor.mean(1,keepdims=True); asd=np.maximum(anchor.std(1,keepdims=True),1e-5)
    return (amu+(raw-rmu)*(asd/rsd)).astype(np.float32)

def blend_residual(anchor,raw_ranker,alpha=ALPHA):
    aligned=align_ranker_to_anchor(raw_ranker,anchor)
    return (anchor+float(alpha)*(aligned-anchor)).astype(np.float32)

def cluster_boot_ci(values,groups,seed,B=BOOTSTRAPS):
    values=np.asarray(values,float); groups=np.asarray(groups,dtype=object)
    ug=np.unique(groups); by={g:np.flatnonzero(groups==g) for g in ug}
    rng=np.random.default_rng(int(seed)); z=np.empty(int(B),float)
    for b in range(int(B)):
        sampled=rng.choice(ug,size=len(ug),replace=True)
        idx=np.concatenate([by[g] for g in sampled])
        z[b]=values[idx].mean()
    return tuple(float(x) for x in np.quantile(z,[.025,.975]))

print("V30-control / V31-IntentState LambdaRank helpers ready")

def _make_cal_local(seed,n_estimators=None):
    return lgb.LGBMClassifier(
        n_estimators=int(LGB_ESTIMATORS if n_estimators is None else n_estimators),
        learning_rate=.03,num_leaves=15,min_child_samples=50,
        reg_alpha=.5,reg_lambda=5.,colsample_bytree=.7,
        random_state=int(seed),n_jobs=4,verbosity=-1
    )


def build_anchor_split_full(train_rows,val_rows,seed):
    train_rows=np.asarray(train_rows,int); val_rows=np.asarray(val_rows,int)
    local_y=y[train_rows]; local_top=Y_TOP[train_rows]; local_bin=Y_BIN[train_rows]
    folds=make_group_folds_local(local_y,TRAIN_GROUP[train_rows],ANCHOR_INNER_FOLDS,seed)
    n=len(train_rows)

    raw=np.zeros((n,N_MODULES),np.float32); top=np.zeros_like(raw)
    for lt,lv in folds:
        mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mr.fit(X[train_rows[lt]],local_y[lt]); mt.fit(X[train_rows[lt]],local_top[lt])
        raw[lv]=mr.predict(X[train_rows[lv]]).astype(np.float32)
        top[lv]=mt.predict(X[train_rows[lv]]).astype(np.float32)
    mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mr.fit(X[train_rows],local_y); mt.fit(X[train_rows],local_top)
    base_o=(.9*raw+.1*top).astype(np.float32)
    base_v=(.9*mr.predict(X[val_rows])+.1*mt.predict(X[val_rows])).astype(np.float32)

    meta_o=np.zeros_like(base_o); meta_v=np.zeros((len(val_rows),N_MODULES),np.float32)
    for m,mid in enumerate(MODULES):
        xt=np.column_stack([base_o[:,m],USER_FEATURES[train_rows]]).astype(np.float32)
        xv=np.column_stack([base_v[:,m],USER_FEATURES[val_rows]]).astype(np.float32)
        yy=local_bin[:,m].astype(np.int32); its=[]
        for f,(lt,lv) in enumerate(folds):
            clf=_make_cal_local(seed+31*m+f)
            clf.fit(xt[lt],yy[lt],eval_set=[(xt[lv],yy[lv])],callbacks=[lgb.early_stopping(LGB_EARLY_STOP,verbose=False)])
            it=int(clf.best_iteration_ or LGB_ESTIMATORS); its.append(it)
            meta_o[lv,m]=clf.predict_proba(xt[lv],num_iteration=it)[:,1].astype(np.float32)
        final_it=max(20,int(np.median(its)))
        full=_make_cal_local(seed+997+31*m,final_it); full.fit(xt,yy)
        meta_v[:,m]=full.predict_proba(xv)[:,1].astype(np.float32)

    cons_o=((1.0-CONS_W)*base_o+CONS_W*meta_o).astype(np.float32)
    cons_v=((1.0-CONS_W)*base_v+CONS_W*meta_v).astype(np.float32)
    return {"base_train":base_o,"meta_train":meta_o,"cons_train":cons_o,
            "base_val":base_v,"meta_val":meta_v,"cons_val":cons_v}

print("Nested frozen anchor helper ready")

STATE_NAMES=np.asarray(["S0","S1","S2","S3"],dtype=object)


def state_codes(pre_anchor,v30,v31):
    pre_top=top5_indices(pre_anchor); v30_top=top5_indices(v30); v31_top=top5_indices(v31)
    a=np.any(pre_top!=v30_top,axis=1)
    b=np.any(v30_top!=v31_top,axis=1)
    return (2*a.astype(np.int8)+b.astype(np.int8)).astype(np.int8)


def top5_sha256(scores):
    top=top5_indices(scores).astype(np.int16)
    return hashlib.sha256(top.tobytes()).hexdigest()


def current066020_surface(v30,v31,state):
    v30=np.asarray(v30,np.float32); v31=np.asarray(v31,np.float32); state=np.asarray(state,np.int8)
    t30=top5_indices(v30); t31=top5_indices(v31)
    top1=(t30[:,0]!=t31[:,0])
    membership=np.asarray([set(a)!=set(b) for a,b in zip(t30,t31)],bool)
    s3=(state==3)
    keep=s3 & (top1 | membership)
    out=v30.copy(); out[keep]=v31[keep]
    info={
        "s3":s3,
        "top1":s3 & top1,
        "membership":s3 & membership & ~top1,
        "order_only":s3 & ~top1 & ~membership,
        "accepted":keep,
    }
    return out,info


def classify_single_adjacent_order_swap(v30_top_row,v31_top_row):
    a=np.asarray(v30_top_row,int); b=np.asarray(v31_top_row,int)
    if a[0]!=b[0] or set(a)!=set(b):
        raise AssertionError(("not pure order-only",a.tolist(),b.tolist()))
    diff=np.flatnonzero(a!=b)
    if len(diff)!=2 or diff[1]!=diff[0]+1:
        raise AssertionError(("not a single adjacent transposition",a.tolist(),b.tolist(),diff.tolist()))
    lo,hi=int(diff[0]),int(diff[1])
    if not (a[lo]==b[hi] and a[hi]==b[lo]):
        raise AssertionError(("not a two-way swap",a.tolist(),b.tolist()))
    boundary=f"B{lo+1}{hi+1}"
    if boundary not in EXPECTED_BOUNDARY_COUNTS:
        raise AssertionError(("unexpected order-only boundary",boundary,a.tolist(),b.tolist()))
    promoted=int(b[lo])
    demoted=int(a[lo])
    return boundary,promoted,demoted,lo+1,hi+1


def submission_from_scores(scores):
    out=sample.copy(); out.loc[:,MODULES]=np.asarray(scores,np.float32)
    assert out.shape==sample.shape
    assert out["user_id"].astype(str).tolist()==test["user_id"].astype(str).tolist()
    assert np.isfinite(out[MODULES].to_numpy()).all()
    return out


def candidate_geometry(name,anchor,candidate):
    ta=top5_indices(anchor); tc=top5_indices(candidate)
    changed=np.any(ta!=tc,axis=1)
    membership=np.asarray([set(a)!=set(b) for a,b in zip(ta,tc)],bool)
    return {
        "candidate":name,
        "changed_users_vs_current":int(changed.sum()),
        "top1_changes":int(np.sum(ta[:,0]!=tc[:,0])),
        "membership_changes":int(membership.sum()),
        "top5_sha256":top5_sha256(candidate),
    }

print("V39.5 state machine + causal boundary helpers ready")


def build_policy_anchor_split(train_rows, val_rows, seed):
    """Fit the frozen structural lineage only on train_rows and predict arbitrary global val_rows."""
    train_rows = np.asarray(train_rows, int)
    val_rows = np.asarray(val_rows, int)
    anc = build_anchor_split_full(train_rows, val_rows, int(seed))

    # Intent-State training features are strictly cross-fitted inside train_rows.
    intent_tr = crossfit_intentstate_features(
        y[train_rows], ORDINAL_GRADES[train_rows], TRAIN_INTENT[train_rows],
        MODULE_READY_BAND[train_rows], TRAIN_GROUP[train_rows], int(seed)+71
    )
    bundle = fit_intentstate_bundle(
        y[train_rows], ORDINAL_GRADES[train_rows], TRAIN_INTENT[train_rows], MODULE_READY_BAND[train_rows]
    )
    intent_va = apply_intentstate_bundle(bundle, ALL_INTENT[val_rows], MODULE_READY_BAND[val_rows])

    Xtr, names, bounds = build_long_matrix_v31(
        train_rows, anc["cons_train"], anc["base_train"], anc["meta_train"], intent_tr
    )
    Xva, _, bounds2 = build_long_matrix_v31(
        val_rows, anc["cons_val"], anc["base_val"], anc["meta_val"], intent_va
    )
    assert bounds == bounds2
    be = bounds["base"]
    ie = bounds["intentstate"]
    _, raw30 = fit_ranker_predict(Xtr[:,:be], Xva[:,:be], ORDINAL_GRADES[train_rows], len(train_rows), int(seed)+101)
    _, raw31 = fit_ranker_predict(Xtr[:,:ie], Xva[:,:ie], ORDINAL_GRADES[train_rows], len(train_rows), int(seed)+102)
    v30 = blend_residual(anc["cons_val"], raw30, ALPHA)
    v31 = blend_residual(anc["cons_val"], raw31, ALPHA)
    st = state_codes(anc["cons_val"], v30, v31)
    current, info = current066020_surface(v30, v31, st)

    # Apply the public-confirmed B34+B45 family mechanically, never by user id.
    t30 = top5_indices(v30); t31 = top5_indices(v31)
    for r in np.flatnonzero(info["order_only"]):
        try:
            boundary, *_ = classify_single_adjacent_order_swap(t30[r], t31[r])
        except AssertionError:
            continue
        if boundary in {"B34", "B45"}:
            current[r] = v31[r]

    # Training structural surface remains purely cross-fitted inside train_rows.
    train_anchor = anc["cons_train"].copy()
    out = {
        "train_anchor": train_anchor,
        "val_anchor": current.astype(np.float32),
        "val_state": st.astype(np.int8),
        "base_train": anc["base_train"], "meta_train": anc["meta_train"],
        "base_val": anc["base_val"], "meta_val": anc["meta_val"],
    }
    del Xtr, Xva, intent_tr, intent_va, bundle, anc, raw30, raw31
    gc.collect()
    return out

print("policy-matched validation anchor helper ready")

print("V42 policy engine initialized; explicit policy_anchor role ready")

POLICY_CACHE=OUT/"policy_anchor_cache"; POLICY_CACHE.mkdir(parents=True,exist_ok=True)
def cached_policy_anchor_split(train_rows,val_rows,seed):
    tr=np.asarray(train_rows,np.int32); va=np.asarray(val_rows,np.int32)
    h=hashlib.sha256(); h.update(b"V131-V42-POLICY-CACHE-v1"); h.update(np.asarray([int(seed)],np.int64).tobytes()); h.update(tr.tobytes()); h.update(b"|"); h.update(va.tobytes())
    f=POLICY_CACHE/(h.hexdigest()+".npz")
    if f.exists():
        z=np.load(f); a=np.asarray(z["val_anchor"],np.float32); st=np.asarray(z["val_state"],np.int8)
        if a.shape==(len(va),M) and st.shape==(len(va),): return {"val_anchor":a,"val_state":st}
    z=build_policy_anchor_split(tr,va,int(seed)); np.savez_compressed(f,val_anchor=z["val_anchor"],val_state=z["val_state"])
    return {"val_anchor":np.asarray(z["val_anchor"],np.float32),"val_state":np.asarray(z["val_state"],np.int8)}
print("policy-anchor cache",POLICY_CACHE)


invalid assessment cells: 251 / 50000
administrative/noise rows: 535 / 14573
USER_FEATURES: (5000, 134)
example augmented text: [older] min aku mau fokus jadi data analyst aja deh mau kuasain excel sama sql tpl_93c96e6b92 topic_excel topic_sql [recent] eh min gajadi deh bosku nyuruh pindah haluan bikin ai ada kelas genai nggak tpl_abc6b29bac topic_genai recent_topic_genai recent_topic_genai [recent] sori nanya mulu ya min tapi kelas yg cover data cleaning ada silabus lengkapnya ga tpl_ace230065e topic_eda recent_topic_eda recent_topic_eda [current] sori nanya mulu ya min tapi kelas belajar pyton dari nol itu berapa lama 
catalog lexical matrices: (5000, 17) (5000, 17)
latest template groups: 1095
latest intent signatures train/test: 46 46
test intent signatures unseen in train: 0
readiness band counts: {np.int8(0): np.int64(45948), np.int8(1): np.int64(16376), np.int8(2): np.int64(5676)}
ordinal grade counts: {np.int16(0): np.int64(48009), np.int16(1): np.int64(1353), np.int16(2): np.i

In [4]:
# =========================
# 3. V129 deterministic views + target-blind validation geometry
# =========================
URL_RE=re.compile(r"https?://\S+|www\.\S+",re.I)
WS_RE=re.compile(r"\s+")
PUNCT_RE=re.compile(r"[^\w<>]+",re.UNICODE)

def norm_text(x):
    x="" if pd.isna(x) else str(x)
    x=html.unescape(unicodedata.normalize("NFKC",x)).casefold()
    x=URL_RE.sub(" <url> ",x); x=PUNCT_RE.sub(" ",x); return WS_RE.sub(" ",x).strip()

ALIASES=[
("generative ai",),("gen ai",),("genai",),("large language model",),("llm",),
("exploratory data analysis",),("eda",),("machine learning",),("ml",),
("computer vision",),("cv",),("natural language processing",),("nlp",),
("prompt engineering",),("web scraping",),("scraping",),("version control",),("git",),
("dashboard",),("mlops",),("no code ai",),("nocode",),("automation",),
("statistics",),("statistik",),("python",),("sql",),("excel",),("career",),("karir",),("portfolio",),("portofolio",)
]
ALIAS_PATTERNS=[re.compile(r"(?<!\w)"+re.escape(a[0])+r"(?!\w)") for a in sorted(ALIASES,key=lambda z:-len(z[0]))]
def skeleton(x):
    s=norm_text(x)
    for rx in ALIAS_PATTERNS: s=rx.sub(" <topic> ",s)
    s=re.sub(r"\b\d+\b"," <num> ",s); return WS_RE.sub(" ",s).strip()

chat=chat.copy(); chat["user_id"]=chat.user_id.astype(str)
chat["timestamp"]=pd.to_datetime(chat.timestamp,errors="coerce")
chat=chat.sort_values(["user_id","timestamp","chat_id"],kind="stable")
byu={u:g for u,g in chat.groupby("user_id",sort=False)}
full_text=[]; latest_text=[]; full_skel=[]; latest_skel=[]; msg_lists=[]; chat_count=[]
for u in ALL_USERS:
    g=byu.get(u)
    msgs=[] if g is None else [norm_text(t) for t in g.user_chat_text.tolist() if norm_text(t)]
    msg_lists.append(msgs); chat_count.append(len(msgs))
    latest=msgs[-1] if msgs else "<no_chat>"; full=" [sep] ".join(msgs) if msgs else "<no_chat>"
    latest_text.append(latest); full_text.append(full); latest_skel.append(skeleton(latest)); full_skel.append(skeleton(full))
full_text=np.array(full_text,dtype=object); latest_text=np.array(latest_text,dtype=object)
full_skel=np.array(full_skel,dtype=object); latest_skel=np.array(latest_skel,dtype=object); chat_count=np.asarray(chat_count,np.float32)

# Assessment parse: invalid/out-of-range values remain NaN until fold-local imputation.
WORD_NUM={"nol":0,"zero":0,"satu":1,"dua":2,"tiga":3,"empat":4,"lima":5}
def parse_obj(s):
    try: return json.loads(s)
    except Exception: return ast.literal_eval(s)
assess=assess.copy(); assess["user_id"]=assess.user_id.astype(str)
assert set(ALL_USERS).issubset(set(assess.user_id))
assess=assess.set_index("user_id").loc[ALL_USERS].reset_index()
objs=[parse_obj(s) for s in assess.assessment_result.astype(str)]
SKILLS=list(objs[0].keys()); assert len(SKILLS)==10
A_RAW=np.full((N,10),np.nan,np.float32); A_BAD=np.zeros((N,10),np.float32)
for i,o in enumerate(objs):
    for j,k in enumerate(SKILLS):
        v=o.get(k,np.nan); z=None
        if isinstance(v,str) and v.strip().casefold() in WORD_NUM: z=float(WORD_NUM[v.strip().casefold()])
        else:
            try: z=float(v)
            except Exception: z=np.nan
        if np.isfinite(z) and abs(z-round(z))<1e-8 and 0<=z<=5: A_RAW[i,j]=z
        else: A_BAD[i,j]=1.0
assert int(A_BAD.sum())==251, f"assessment parser drift: {A_BAD.sum()} invalid cells"

# Module documents and readiness map from organizer catalog semantics.
mc=modules.set_index("module_id").loc[MODULES]
MOD_TEXT=(mc.module_name.astype(str)+" [sep] "+mc.description_and_syllabus.astype(str)+" [sep] "+mc.prerequisite_level.astype(str)).map(norm_text).tolist()
MOD_SKILLS={
0:[],1:[0],2:[1],3:[0],4:[9],5:[2],6:[0,2,3],7:[1],8:[2,3,4,5],9:[4,6],10:[4],11:[4,7],12:[7],13:[7,8],14:[7],15:[0,4,5,9],16:[8,9]
}

# Target-blind near-template components. No-chat users are deliberately unique groups.
vec_grp=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=1,max_features=50000,norm="l2")
Xg=vec_grp.fit_transform(full_skel.tolist())
sim=(Xg@Xg.T).tocsr(); sim.data=(sim.data>=cfg.near_threshold).astype(np.int8); sim.eliminate_zeros()
_,near_group=connected_components(sim,directed=False)
nochat=np.array([t=="<no_chat>" for t in full_text])
base_gid=int(near_group.max())+1
near_group=near_group.astype(np.int64); near_group[nochat]=base_gid+np.flatnonzero(nochat)
ALL_TRAIN=np.arange(NTR,dtype=int)
ALL_TEST=np.arange(NTR,N,dtype=int)

# Manifest is target-free. It is diagnostic only and never used as a test label proxy.
pd.DataFrame({
    "user_id":train.user_id.astype(str),
    "near_group":near_group[:NTR],
    "latest_skeleton":latest_skel[:NTR],
    "full_history_sha256":[hashlib.sha256(str(t).encode()).hexdigest() for t in full_text[:NTR]],
}).to_csv(OUT/"v131_template_manifest.csv",index=False)
print("template geometry ready; near groups(train/all)=",len(np.unique(near_group[:NTR])),len(np.unique(near_group)))
del Xg,sim,vec_grp; gc.collect()


template geometry ready; near groups(train/all)= 1912 2297


0

In [5]:

# =========================
# 4. Frozen E5 + Nomic representations, pinned revisions and legacy-cache probe verification
# =========================
E5_ID="intfloat/multilingual-e5-base"; E5_REV="d128750597153bb5987e10b1c3493a34e5a4502a"
NOMIC_ID="nomic-ai/nomic-embed-text-v2-moe"; NOMIC_REV="1066b6599d099fbb93dfcb64f9c37a7c9e503e85"
NOMIC_CODE_REV="7710840340a098cfb869c4f65e87cf2b1b70caca"

def text_sha(texts,prefix,model_id):
    h=hashlib.sha256(); h.update((model_id+"|"+prefix).encode())
    for t in texts: h.update(b"\0"); h.update(str(t).encode("utf-8"))
    return h.hexdigest()

def _resolve_model(model_id):
    needle=model_id.split('/')[-1].lower().replace('_','-')
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob('config.json'):
            s=str(p.parent).lower().replace('_','-')
            if needle in s: return str(p.parent)
    return model_id

def _external_pair(name,kind):
    if not Path('/kaggle/input').exists(): return None
    target=f"{name}_{kind}.npy"
    for f in Path('/kaggle/input').rglob(target):
        meta=f.with_suffix('.json')
        if meta.exists(): return f,meta
    return None

def _legacy_ok(pair,texts,prefix,model_id):
    if pair is None: return None
    f,meta=pair
    try:
        m=json.loads(meta.read_text()); arr=np.load(f,mmap_mode='r')
        legacy=text_sha(texts,prefix,model_id)
        if m.get('key')!=legacy or arr.shape[0]!=len(texts): return None
        return np.asarray(arr,np.float32)
    except Exception: return None

def _cos_rows(a,b):
    a=np.asarray(a,np.float32); b=np.asarray(b,np.float32)
    a=a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12); b=b/np.maximum(np.linalg.norm(b,axis=1,keepdims=True),1e-12)
    return np.sum(a*b,axis=1)

def load_e5_pair(user_texts,module_texts,device):
    up=_legacy_ok(_external_pair('users','e5'),user_texts,'query: ',E5_ID)
    mp=_legacy_ok(_external_pair('modules','e5'),module_texts,'passage: ',E5_ID)
    import torch
    from sentence_transformers import SentenceTransformer
    mid=_resolve_model(E5_ID); model=SentenceTransformer(mid,device=device,revision=None if mid!=E5_ID else E5_REV)
    probes_u=[0,min(123,len(user_texts)-1),len(user_texts)-1]; probes_m=[0,min(5,len(module_texts)-1),len(module_texts)-1]
    if up is not None and mp is not None:
        pu=np.asarray(model.encode(['query: '+user_texts[i] for i in probes_u],batch_size=8,normalize_embeddings=True,convert_to_numpy=True),np.float32)
        pm=np.asarray(model.encode(['passage: '+module_texts[i] for i in probes_m],batch_size=8,normalize_embeddings=True,convert_to_numpy=True),np.float32)
        if float(_cos_rows(pu,up[probes_u]).min())>.999 and float(_cos_rows(pm,mp[probes_m]).min())>.999:
            print('Reusing probe-verified external E5 cache')
            del model; gc.collect(); torch.cuda.empty_cache(); return up,mp
    print('Encoding pinned E5 revision',E5_REV)
    up=np.asarray(model.encode(['query: '+t for t in user_texts],batch_size=64,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True),np.float32)
    mp=np.asarray(model.encode(['passage: '+t for t in module_texts],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True),np.float32)
    np.save(OUT/'users_e5.npy',up); np.save(OUT/'modules_e5.npy',mp)
    json.dump({'model':E5_ID,'revision':E5_REV,'user_key':text_sha(user_texts,'query: ',E5_ID),'module_key':text_sha(module_texts,'passage: ',E5_ID),'shape_users':list(up.shape),'shape_modules':list(mp.shape)},open(OUT/'e5_cache_meta.json','w'),indent=2)
    del model; gc.collect(); torch.cuda.empty_cache(); return up,mp

def _nomic_encode(model,tok,texts,prefix,device,batch=48):
    import torch
    rows=[]
    with torch.no_grad():
        for a in range(0,len(texts),batch):
            e=tok([prefix+t for t in texts[a:a+batch]],padding=True,truncation=True,max_length=256,return_tensors='pt'); e={k:v.to(device) for k,v in e.items()}
            h=model(**e).last_hidden_state; mask=e['attention_mask'].unsqueeze(-1).float()
            v=((h*mask).sum(1)/mask.sum(1).clamp_min(1)).float(); v=v/v.norm(dim=1,keepdim=True).clamp_min(1e-12); rows.append(v.cpu().numpy())
    return np.concatenate(rows).astype(np.float32)

def load_nomic_pair(user_texts,module_texts,device):
    up=_legacy_ok(_external_pair('users','nomic'),user_texts,'search_query: ',NOMIC_ID)
    mp=_legacy_ok(_external_pair('modules','nomic'),module_texts,'search_document: ',NOMIC_ID)
    import torch
    from transformers import AutoTokenizer,AutoModel
    mid=_resolve_model(NOMIC_ID)
    tok=AutoTokenizer.from_pretrained(mid,trust_remote_code=True,revision=None if mid!=NOMIC_ID else NOMIC_REV)
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    kw=dict(trust_remote_code=True,torch_dtype=dtype)
    if mid==NOMIC_ID: kw.update(revision=NOMIC_REV,code_revision=NOMIC_CODE_REV)
    model=AutoModel.from_pretrained(mid,**kw).to(device).eval()
    probes_u=[0,min(123,len(user_texts)-1),len(user_texts)-1]; probes_m=[0,min(5,len(module_texts)-1),len(module_texts)-1]
    if up is not None and mp is not None:
        pu=_nomic_encode(model,tok,[user_texts[i] for i in probes_u],'search_query: ',device,batch=8)
        pm=_nomic_encode(model,tok,[module_texts[i] for i in probes_m],'search_document: ',device,batch=8)
        if float(_cos_rows(pu,up[probes_u]).min())>.999 and float(_cos_rows(pm,mp[probes_m]).min())>.999:
            print('Reusing probe-verified external Nomic cache')
            del model; gc.collect(); torch.cuda.empty_cache(); return up,mp
    print('Encoding pinned Nomic revision',NOMIC_REV,'code',NOMIC_CODE_REV)
    up=_nomic_encode(model,tok,user_texts,'search_query: ',device); mp=_nomic_encode(model,tok,module_texts,'search_document: ',device,batch=17)
    np.save(OUT/'users_nomic.npy',up); np.save(OUT/'modules_nomic.npy',mp)
    json.dump({'model':NOMIC_ID,'revision':NOMIC_REV,'code_revision':NOMIC_CODE_REV,'user_key':text_sha(user_texts,'search_query: ',NOMIC_ID),'module_key':text_sha(module_texts,'search_document: ',NOMIC_ID),'shape_users':list(up.shape),'shape_modules':list(mp.shape)},open(OUT/'nomic_cache_meta.json','w'),indent=2)
    del model; gc.collect(); torch.cuda.empty_cache(); return up,mp

import torch
if not torch.cuda.is_available(): dev_e5=dev_no='cpu'; print('WARNING CUDA unavailable; production target is T4x2')
else:
    ng=torch.cuda.device_count(); dev_e5='cuda:0'; dev_no='cuda:1' if ng>1 else 'cuda:0'
E5U,E5M=load_e5_pair(full_text.tolist(),MOD_TEXT,dev_e5)
NOU,NOM=load_nomic_pair(full_text.tolist(),MOD_TEXT,dev_no)
assert E5U.shape==(N,768) and E5M.shape[0]==M and NOU.shape[0]==N and NOM.shape[0]==M
E5_MOD=(E5U@E5M.T).astype(np.float32); NOM_MOD=(NOU@NOM.T).astype(np.float32)
print('dense ready',E5U.shape,NOU.shape)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

configuration_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.90G [00:00<?, ?B/s]

Encoding pinned Nomic revision 1066b6599d099fbb93dfcb64f9c37a7c9e503e85 code 7710840340a098cfb869c4f65e87cf2b1b70caca
dense ready (5000, 768) (5000, 768)


In [6]:
# =========================
# 5. Lean V129 candidate features with explicit retrieval_anchor geometry
# =========================
def fit_bundle(ref_idx):
    ref_idx=np.asarray(ref_idx,int)
    corpus=[full_text[i] for i in ref_idx]+MOD_TEXT
    wv=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=cfg.word_max_features,sublinear_tf=True,smooth_idf=True,norm="l2")
    cv=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,max_features=cfg.char_max_features,sublinear_tf=True,norm="l2")
    wv.fit(corpus); cv.fit(corpus)
    return {
        "wv":wv,"cv":cv,
        "wfull":wv.transform(full_text.tolist()),"wlatest":wv.transform(latest_text.tolist()),"wm":wv.transform(MOD_TEXT),
        "cfull":cv.transform(full_text.tolist()),"clatest":cv.transform(latest_text.tolist()),"cm":cv.transform(MOD_TEXT)
    }

def recency_similarity(bundle,indices,key="word"):
    indices=np.asarray(indices,int); out=np.zeros((len(indices),M),np.float32)
    vec=bundle["wv"] if key=="word" else bundle["cv"]
    mm=bundle["wm"] if key=="word" else bundle["cm"]
    for qi,g in enumerate(indices):
        msgs=msg_lists[g]
        if not msgs: continue
        S=(vec.transform(msgs)@mm.T).toarray().astype(np.float32)
        w=np.power(0.5,np.arange(len(msgs)-1,-1,-1,dtype=np.float32)); w/=w.sum()
        out[qi]=(w[:,None]*S).sum(0)
    return out

def geom_from_scores(S,prefix):
    S=np.asarray(S,np.float32); order=np.argsort(-S,axis=1,kind="stable"); ranks=np.empty_like(order)
    rr=np.arange(len(S))[:,None]; ranks[rr,order]=np.arange(1,M+1)[None,:]
    best=S.max(1,keepdims=True); fifth=np.take_along_axis(S,order[:,4:5],1)
    return [(prefix+"_score",S),(prefix+"_rank",ranks.astype(np.float32)/M),(prefix+"_gapbest",best-S),(prefix+"_gap5",S-fifth)]

def impute_assessment(ref_idx,q_idx):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int)
    med=np.nanmedian(A_RAW[ref_idx],axis=0); med=np.where(np.isfinite(med),med,2.5)
    A=np.where(np.isfinite(A_RAW[q_idx]),A_RAW[q_idx],med[None,:]).astype(np.float32)
    return A,A_BAD[q_idx].astype(np.float32)

def readiness(A,BAD):
    R=np.zeros((len(A),M),np.float32); I=np.zeros_like(R)
    for m,js in MOD_SKILLS.items():
        if js: R[:,m]=A[:,js].mean(1); I[:,m]=BAD[:,js].mean(1)
        else: R[:,m]=A.mean(1); I[:,m]=BAD.mean(1)
    return R,I

def candidate_features(bundle,ref_idx,q_idx,retrieval_anchor_q):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int); pieces=[]
    wf=(bundle["wfull"][q_idx]@bundle["wm"].T).toarray().astype(np.float32)
    wl=(bundle["wlatest"][q_idx]@bundle["wm"].T).toarray().astype(np.float32)
    wr=recency_similarity(bundle,q_idx,"word")
    cf=(bundle["cfull"][q_idx]@bundle["cm"].T).toarray().astype(np.float32)
    cl=(bundle["clatest"][q_idx]@bundle["cm"].T).toarray().astype(np.float32)
    cr=recency_similarity(bundle,q_idx,"char")
    for p,S in [
        ("tfidf_word_full",wf),("tfidf_word_latest",wl),("tfidf_word_recency",wr),
        ("tfidf_char_full",cf),("tfidf_char_latest",cl),("tfidf_char_recency",cr),
        ("e5_mod",E5_MOD[q_idx]),("nomic_mod",NOM_MOD[q_idx]),
    ]: pieces.extend(geom_from_scores(S,p))
    pieces += [("dense_e5_minus_nomic",E5_MOD[q_idx]-NOM_MOD[q_idx])]

    Aq,Bq=impute_assessment(ref_idx,q_idx); R,I=readiness(Aq,Bq)
    pieces += [("ready",R),("ready_invalid",I)]
    for j,s in enumerate(SKILLS):
        pieces.append((f"assess_{j:02d}",np.repeat(Aq[:,j:j+1],M,1)))
        pieces.append((f"assess_bad_{j:02d}",np.repeat(Bq[:,j:j+1],M,1)))
    pieces += [
        ("assess_mean",np.repeat(Aq.mean(1,keepdims=True),M,1)),
        ("assess_std",np.repeat(Aq.std(1,keepdims=True),M,1)),
        ("assess_invalid_count",np.repeat(Bq.sum(1,keepdims=True),M,1)),
        ("chat_count",np.repeat(chat_count[q_idx,None],M,1)),
        ("no_chat",np.repeat((chat_count[q_idx]==0).astype(np.float32)[:,None],M,1)),
    ]
    pieces.extend(geom_from_scores(retrieval_anchor_q,"retrieval_anchor"))

    names=[]; arrays=[]
    for n,a in pieces:
        a=np.asarray(a,np.float32); assert a.shape==(len(q_idx),M),(n,a.shape)
        names.append(n); arrays.append(a[:,:,None])
    X=np.concatenate(arrays,2).astype(np.float32)
    assert np.isfinite(X).all()
    # V129 contract: no BM25 and no target-derived KNN/template statistics.
    assert not any(("bm25" in n) or n.startswith("knn_") or n.startswith("tpl_") for n in names)
    return X,names


In [7]:
# =========================
# 6. V129 Stage-1: retrieval_anchor + per-module LightGBM
# =========================
def make_inner_splits(indices,seed):
    indices=np.asarray(indices,int); groups=near_group[indices]; strat=np.argmax(y[indices],1)
    try:
        sg=StratifiedGroupKFold(cfg.n_inner,shuffle=True,random_state=seed)
        return [(indices[a],indices[b]) for a,b in sg.split(np.zeros(len(indices)),strat,groups)]
    except Exception:
        gg=GroupKFold(cfg.n_inner)
        return [(indices[a],indices[b]) for a,b in gg.split(np.zeros(len(indices)),groups=groups)]

def _anchor_matrix(bundle,ref_idx,q_idx,med_ref=None):
    ref_idx=np.asarray(ref_idx,int); q_idx=np.asarray(q_idx,int)
    if med_ref is None:
        med_ref=np.nanmedian(A_RAW[ref_idx],axis=0); med_ref=np.where(np.isfinite(med_ref),med_ref,2.5)
    Aq=np.where(np.isfinite(A_RAW[q_idx]),A_RAW[q_idx],med_ref[None,:]).astype(np.float32)
    B=A_BAD[q_idx].astype(np.float32)
    extra=np.c_[Aq,B,chat_count[q_idx],(chat_count[q_idx]==0).astype(np.float32)]
    return hstack([bundle["wfull"][q_idx],bundle["cfull"][q_idx],csr_matrix(extra)],format="csr")

def nested_anchor(bundle,train_idx,query_idx,inner_splits):
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int); pos={g:i for i,g in enumerate(train_idx)}
    oof_rel=np.zeros((len(train_idx),M),np.float32); oof_top=np.zeros_like(oof_rel)
    for f,(tr,va) in enumerate(inner_splits):
        med=np.nanmedian(A_RAW[tr],axis=0); med=np.where(np.isfinite(med),med,2.5)
        Xtr=_anchor_matrix(bundle,tr,tr,med); Xv=_anchor_matrix(bundle,tr,va,med)
        r=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,y[tr])
        t=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,np.eye(M,dtype=np.float32)[np.argmax(y[tr],1)])
        loc=np.array([pos[g] for g in va]); oof_rel[loc]=r.predict(Xv); oof_top[loc]=t.predict(Xv)
    med=np.nanmedian(A_RAW[train_idx],axis=0); med=np.where(np.isfinite(med),med,2.5)
    Xtr=_anchor_matrix(bundle,train_idx,train_idx,med); Xq=_anchor_matrix(bundle,train_idx,query_idx,med)
    r=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,y[train_idx])
    t=Ridge(alpha=3.0,solver="lsqr",tol=2e-3,max_iter=1000).fit(Xtr,np.eye(M,dtype=np.float32)[np.argmax(y[train_idx],1)])
    oof=(.9*oof_rel+.1*oof_top).astype(np.float32); q=(.9*r.predict(Xq)+.1*t.predict(Xq)).astype(np.float32)
    return oof,q

def fit_lgb_stage1(Xtr,ytr,Xq):
    assert Xtr.ndim==3 and Xq.ndim==3 and Xtr.shape[1]==Xq.shape[1]==M and Xtr.shape[2]==Xq.shape[2]
    pred=np.zeros((len(Xq),M),np.float32)
    for m in range(M):
        vals=[]
        for seed in cfg.lgb_seeds:
            model=lgb.LGBMRegressor(
                objective="regression_l2",n_estimators=cfg.lgb_estimators,learning_rate=cfg.lgb_lr,
                num_leaves=cfg.lgb_num_leaves,max_depth=cfg.lgb_max_depth,min_child_samples=cfg.lgb_min_child,
                subsample=cfg.lgb_subsample,subsample_freq=1,colsample_bytree=cfg.lgb_colsample,
                reg_alpha=cfg.lgb_reg_alpha,reg_lambda=cfg.lgb_reg_lambda,
                random_state=int(seed+37*m),n_jobs=2,verbosity=-1,force_col_wise=True)
            model.fit(Xtr[:,m,:],ytr[:,m]); vals.append(model.predict(Xq[:,m,:]))
        pred[:,m]=np.mean(vals,axis=0)
    return pred

def fit_stage1_once(train_idx,query_idx,bundle,seed):
    """Train Stage-1 without using any query labels. `bundle` may include query text vocabulary only label-free."""
    train_idx=np.asarray(train_idx,int); query_idx=np.asarray(query_idx,int)
    inner=make_inner_splits(train_idx,seed+11)
    anchor_tr,anchor_q=nested_anchor(bundle,train_idx,query_idx,inner)
    Xtr,names=candidate_features(bundle,train_idx,train_idx,anchor_tr)
    Xq,nq=candidate_features(bundle,train_idx,query_idx,anchor_q); assert nq==names
    pred=fit_lgb_stage1(Xtr,y[train_idx],Xq)
    assert np.isfinite(pred).all()
    del Xtr; gc.collect()
    return {"retrieval_anchor_query":anchor_q,"fusion_query":pred,"raw_query":Xq,"names":names}

def make_world_splits(indices,world,seed=SEED):
    indices=np.asarray(indices,int)
    strat=np.array([f"{np.argmax(y[i])}_{int((y[i]>0).sum())}" for i in indices],object)
    cc=Counter(strat); strat=np.array([s if cc[s]>=cfg.n_outer else str(np.argmax(y[i])) for s,i in zip(strat,indices)],object)
    if world=="random":
        sk=StratifiedKFold(cfg.n_outer,shuffle=True,random_state=seed)
        return [(indices[a],indices[b]) for a,b in sk.split(np.zeros(len(indices)),strat)]
    if world in {"exact","latest","near"}:
        groups={"exact":full_text[indices],"latest":latest_skel[indices],"near":near_group[indices]}[world]
        try:
            sg=StratifiedGroupKFold(cfg.n_outer,shuffle=True,random_state=seed)
            return [(indices[a],indices[b]) for a,b in sg.split(np.zeros(len(indices)),strat,groups)]
        except Exception:
            gg=GroupKFold(cfg.n_outer)
            return [(indices[a],indices[b]) for a,b in gg.split(np.zeros(len(indices)),groups=groups)]
    if world=="contiguous":
        order=indices[np.argsort([int(re.sub(r"\D","",ALL_USERS[i])) for i in indices])]
        blocks=np.array_split(order,cfg.n_outer)
        return [(np.setdiff1d(indices,b),np.asarray(b,int)) for b in blocks]
    raise ValueError(world)


In [8]:

# =========================
# 7. Legacy V129/V130 analogue action head + V131 global BM25 action representation
# =========================
def action_table(indices,policy_anchor,fusion,raw,names):
    indices=np.asarray(indices,int); policy_anchor=np.asarray(policy_anchor,np.float32); n=len(indices)
    X=[]; Y=[]; users=[]; cand=[]; inc=[]; ranks=[]
    order=np.argsort(-policy_anchor,axis=1,kind='stable')
    for i,g in enumerate(indices):
        im=int(order[i,0])
        for rr in (1,2,3):
            cm=int(order[i,rr]); s=policy_anchor[i].copy()
            if g<NTR:
                before=per_user_ndcg(y[g:g+1],s[None,:])[0]; s[im],s[cm]=s[cm],s[im]
                delta=float(per_user_ndcg(y[g:g+1],s[None,:])[0]-before)
            else: delta=0.0
            diff=raw[i,cm]-raw[i,im]
            pair=np.zeros(M*M,np.float32); pair[im*M+cm]=1.0
            c1=np.zeros(M,np.float32); c1[cm]=1.0
            i1=np.zeros(M,np.float32); i1[im]=1.0
            r1=np.zeros(3,np.float32); r1[rr-1]=1.0
            feat=np.r_[fusion[i,cm],fusion[i,im],fusion[i,cm]-fusion[i,im],policy_anchor[i,im]-policy_anchor[i,cm],diff,c1,i1,pair,r1]
            X.append(feat); Y.append(delta); users.append(i); cand.append(cm); inc.append(im); ranks.append(rr+1)
    fn=['fusion_c','fusion_i','fusion_diff','policy_anchor_margin']+['d_'+x for x in names]+[f'cand_{m}' for m in MODULES]+[f'inc_{m}' for m in MODULES]+[f'pair_{a}_{b}' for a in MODULES for b in MODULES]+['rank2','rank3','rank4']
    X=np.asarray(X,np.float32); Y=np.asarray(Y,np.float32)
    assert X.shape[0]==3*n and np.isfinite(X).all() and np.isfinite(Y).all()
    return X,Y,np.asarray(users,int),np.asarray(cand,int),np.asarray(inc,int),np.asarray(ranks,int),fn

def fit_action_model(X,Y):
    model=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000)); model.fit(X,Y); return model

def predict_action(model,indices,policy_anchor,fusion,raw,names):
    X,Y,U,C,I,R,fn=action_table(indices,policy_anchor,fusion,raw,names)
    return {'X':X,'y':Y,'user':U,'cand':C,'inc':I,'rank':R,'pred':model.predict(X).astype(np.float32),'features':fn}

def bootstrap_ci(d,groups=None,seed=SEED):
    d=np.asarray(d,float); rng=np.random.default_rng(seed); vals=np.empty(cfg.bootstraps,float)
    if groups is None:
        n=len(d)
        for b in range(cfg.bootstraps): vals[b]=d[rng.integers(0,n,n)].mean()
    else:
        groups=np.asarray(groups); ug=np.unique(groups); bucket={g:np.flatnonzero(groups==g) for g in ug}
        for b in range(cfg.bootstraps):
            pick=rng.choice(ug,len(ug),replace=True); ix=np.concatenate([bucket[g] for g in pick]); vals[b]=d[ix].mean()
    return float(np.quantile(vals,.025)),float(np.quantile(vals,.975))

def choose_user_actions(pred,y_delta,cand,inc,n_users):
    P=np.asarray(pred).reshape(n_users,3); D=np.asarray(y_delta).reshape(n_users,3); C=np.asarray(cand).reshape(n_users,3); I=np.asarray(inc).reshape(n_users,3)
    j=np.argmax(P,1); rr=np.arange(n_users); return P[rr,j],D[rr,j],C[rr,j],I[rr,j],j

def select_fixed_cap(score,realized,candidate,eligible,cap):
    score=np.asarray(score,float); realized=np.asarray(realized,float); eligible=np.asarray(eligible,bool)
    ids=np.flatnonzero((score>0)&eligible); pick=ids[np.argsort(-score[ids],kind='stable')[:min(int(cap),len(ids))]]
    sel=np.zeros(len(score),bool); sel[pick]=True; ud=np.zeros(len(score),float); ud[pick]=realized[pick]
    return sel,ud,pick

def categorical_tv(a,b):
    a=pd.Series(list(a),dtype='object').value_counts(normalize=True); b=pd.Series(list(b),dtype='object').value_counts(normalize=True)
    idx=a.index.union(b.index); return float(0.5*np.abs(a.reindex(idx,fill_value=0)-b.reindex(idx,fill_value=0)).sum())

def smd(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float); den=np.sqrt((np.var(a)+np.var(b))/2.0)
    return float((np.mean(b)-np.mean(a))/max(den,1e-12))

assert abs(ndcg5(y[:5],y[:5])-1.0)<1e-8
print('action utility helpers ready; hard cap is magnitude-aligned, no benefit:harm purity gate')


# -------------------------
# V131 target-blind lexical complement: exact V128 BM25 definition.
# -------------------------
class BM25Index:
    def __init__(self,docs,ngram=(1,1),k1=1.2,b=0.75,max_features=70000):
        self.k1=float(k1); self.b=float(b)
        self.vec=CountVectorizer(token_pattern=r"(?u)\b\w+\b",ngram_range=ngram,max_features=max_features)
        c=self.vec.fit_transform(list(docs)).tocsr().astype(np.float32)
        self.n=c.shape[0]
        df=np.asarray((c>0).sum(0)).ravel()
        self.idf=np.log(1+(self.n-df+.5)/(df+.5)).astype(np.float32)
        dl=np.asarray(c.sum(1)).ravel(); avg=max(float(dl.mean()),1e-6)
        d=c.copy()
        for i in range(self.n):
            a,z=d.indptr[i],d.indptr[i+1]
            tf,cols=d.data[a:z],d.indices[a:z]
            den=tf+self.k1*(1-self.b+self.b*dl[i]/avg)
            d.data[a:z]=tf*(self.k1+1)/np.maximum(den,1e-12)*self.idf[cols]
        self.d=d
    def score(self,queries):
        return np.asarray((self.vec.transform(list(queries)).astype(np.float32)@self.d.T).toarray(),np.float32)

BM25_UNI=BM25Index(MOD_TEXT,(1,1),1.2,0.75,70000)
BM25_BI=BM25Index(MOD_TEXT,(1,2),1.2,0.75,70000)
BM25_UNI_FULL=BM25_UNI.score(full_text.tolist())
BM25_UNI_LATEST=BM25_UNI.score(latest_text.tolist())
BM25_BI_FULL=BM25_BI.score(full_text.tolist())
BM25_BI_LATEST=BM25_BI.score(latest_text.tolist())
BM25_UNI_RECENCY=np.zeros((N,M),np.float32)
for g,msgs in enumerate(msg_lists):
    if not msgs: continue
    s=BM25_UNI.score(msgs)
    w=np.power(0.5,np.arange(len(msgs)-1,-1,-1,dtype=np.float32)); w/=w.sum()
    BM25_UNI_RECENCY[g]=(w[:,None]*s).sum(0)
assert all(np.isfinite(z).all() for z in [BM25_UNI_FULL,BM25_UNI_LATEST,BM25_UNI_RECENCY,BM25_BI_FULL,BM25_BI_LATEST])

# Fixed label-free state indicators used by the post-V130 research probe.
NEG_RX=re.compile(r"\b(?:tidak|ga|gak|nggak|enggak|bukan|jangan|tanpa|belum|skip|hapus|cancel|batal)\b",re.I)
COMP_RX=re.compile(r"\b(?:atau|vs|versus|dibanding|daripada|mending|lebih cocok|bedanya|beda)\b",re.I)
NEG_FLAG=np.asarray([bool(NEG_RX.search(str(t))) for t in full_text],np.float32)
COMP_FLAG=np.asarray([bool(COMP_RX.search(str(t))) for t in full_text],np.float32)
TOPIC_PRESENT=(MODULE_COUNT>0).astype(np.float32)
LATEST_TOPIC_PRESENT=(MODULE_LATEST>0).astype(np.float32)
LATEST_FULL_CONTRADICTION=np.asarray([
    bool(TOPIC_PRESENT[i].any() and LATEST_TOPIC_PRESENT[i].any() and np.any(TOPIC_PRESENT[i]!=LATEST_TOPIC_PRESENT[i]))
    for i in range(N)
],np.float32)

def _nranks(S):
    S=np.asarray(S,np.float32); order=np.argsort(-S,axis=1,kind='stable'); ranks=np.empty_like(order)
    ranks[np.arange(len(S))[:,None],order]=np.arange(1,M+1)[None,:]
    return ranks.astype(np.float32)/M

def _score_action_block(S,rows,inc,cand,prefix):
    S=np.asarray(S,np.float32); rows=np.asarray(rows,int); inc=np.asarray(inc,int); cand=np.asarray(cand,int)
    R=_nranks(S)
    X=np.c_[S[rows,cand],S[rows,inc],S[rows,cand]-S[rows,inc],R[rows,cand],R[rows,inc]].astype(np.float32)
    names=[prefix+'_c',prefix+'_i',prefix+'_diff',prefix+'_rank_c',prefix+'_rank_i']
    return X,names

def _global_view_matrices(bundle,q_idx):
    q_idx=np.asarray(q_idx,int)
    wf=(bundle['wfull'][q_idx]@bundle['wm'].T).toarray().astype(np.float32)
    wl=(bundle['wlatest'][q_idx]@bundle['wm'].T).toarray().astype(np.float32)
    wr=recency_similarity(bundle,q_idx,'word')
    cf=(bundle['cfull'][q_idx]@bundle['cm'].T).toarray().astype(np.float32)
    cl=(bundle['clatest'][q_idx]@bundle['cm'].T).toarray().astype(np.float32)
    cr=recency_similarity(bundle,q_idx,'char')
    return {
        'tfidf_word_full':wf,'tfidf_word_latest':wl,'tfidf_word_recency':wr,
        'tfidf_char_full':cf,'tfidf_char_latest':cl,'tfidf_char_recency':cr,
        'e5':E5_MOD[q_idx],'nomic':NOM_MOD[q_idx],
        'bm25_uni_full':BM25_UNI_FULL[q_idx],'bm25_uni_latest':BM25_UNI_LATEST[q_idx],
        'bm25_uni_recency':BM25_UNI_RECENCY[q_idx],'bm25_bi_full':BM25_BI_FULL[q_idx],'bm25_bi_latest':BM25_BI_LATEST[q_idx],
    }

def global_action_table(indices,policy_anchor,bundle,ref_idx):
    """Primary V131 row contract: global pooling, no incumbent×challenger pair one-hot."""
    indices=np.asarray(indices,int); ref_idx=np.asarray(ref_idx,int); n=len(indices)
    order=np.argsort(-np.asarray(policy_anchor,np.float32),axis=1,kind='stable')
    inc=np.repeat(order[:,0],3); cand=order[:,1:4].reshape(-1); rank=np.tile(np.array([2,3,4],np.int8),n)
    grow=np.repeat(indices,3); qloc=np.repeat(np.arange(n),3)

    c1=np.eye(M,dtype=np.float32)[cand]; i1=np.eye(M,dtype=np.float32)[inc]
    r1=np.eye(3,dtype=np.float32)[rank-2]
    parts=[c1,i1,r1]; names=[f'cand_{m}' for m in MODULES]+[f'inc_{m}' for m in MODULES]+['rank2','rank3','rank4']

    views=_global_view_matrices(bundle,indices)
    for key in ['tfidf_word_full','tfidf_word_latest','tfidf_word_recency','tfidf_char_full','tfidf_char_latest','tfidf_char_recency','e5','nomic',
                'bm25_uni_full','bm25_uni_latest','bm25_uni_recency','bm25_bi_full','bm25_bi_latest']:
        S=views[key]; R=_nranks(S); cc=S[qloc,cand]; ii=S[qloc,inc]
        parts.append(np.c_[cc,ii,cc-ii,R[qloc,cand],R[qloc,inc]].astype(np.float32))
        names += [key+'_c',key+'_i',key+'_diff',key+'_rank_c',key+'_rank_i']

    Aq,Bq=impute_assessment(ref_idx,indices); ready_q,_=readiness(Aq,Bq)
    rc=ready_q[qloc,cand]; ri=ready_q[qloc,inc]
    state=np.c_[
        rc,ri,rc-ri,
        np.repeat(Aq.mean(1),3),np.repeat(Aq.std(1),3),np.repeat(Bq.sum(1),3),
        np.repeat(np.log1p(chat_count[indices]),3),np.repeat((chat_count[indices]==0).astype(np.float32),3),
        np.repeat(NEG_FLAG[indices],3),np.repeat(COMP_FLAG[indices],3),np.repeat(LATEST_FULL_CONTRADICTION[indices],3),
        TOPIC_PRESENT[grow,cand],TOPIC_PRESENT[grow,inc]
    ].astype(np.float32)
    parts.append(state); names += ['ready_c','ready_i','ready_diff','assess_mean','assess_std','assess_invalid_count','log_chat_count','no_chat','negation','comparison','latest_full_contradiction','candidate_explicit','incumbent_explicit']

    X=np.c_[*parts].astype(np.float32)
    Y=np.zeros(3*n,np.float32)
    for i,g in enumerate(indices):
        base=np.asarray(policy_anchor[i],np.float32)
        before=per_user_ndcg(y[g:g+1],base[None,:])[0] if g<NTR else 0.0
        for j in range(3):
            if g>=NTR: continue
            s=base.copy(); a=int(order[i,0]); b=int(order[i,j+1]); s[a],s[b]=s[b],s[a]
            Y[3*i+j]=float(per_user_ndcg(y[g:g+1],s[None,:])[0]-before)
    assert X.shape[0]==3*n and len(names)==X.shape[1] and np.isfinite(X).all() and np.isfinite(Y).all()
    assert not any(x.startswith('pair_') for x in names)
    return {'X':X,'y':Y,'inc':inc.astype(np.int16),'cand':cand.astype(np.int16),'rank':rank,'features':names}

def feature_mask(names,variant):
    names=list(names); keep=np.ones(len(names),bool)
    bm=np.array([x.startswith('bm25_') for x in names],bool)
    if variant=='primary': pass
    elif variant=='no_bm25': keep &= ~bm
    elif variant=='bm25_full':
        keep &= ~(bm & np.array([not (x.startswith('bm25_uni_full_') or x.startswith('bm25_bi_full_')) for x in names],bool))
    elif variant=='bm25_temporal':
        keep &= ~(bm & np.array([not (x.startswith('bm25_uni_latest_') or x.startswith('bm25_uni_recency_') or x.startswith('bm25_bi_latest_')) for x in names],bool))
    else: raise ValueError(variant)
    return keep

def fit_global_action_model(X,Y,names,variant='primary'):
    mask=feature_mask(names,variant)
    model=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000))
    model.fit(X[:,mask],Y)
    return model,mask

def predict_global_action(model,mask,table):
    return model.predict(table['X'][:,mask]).astype(np.float32).reshape(-1,3)

def select_rank2_fixed_cap(pred3,y3,eligible,cap):
    score=np.asarray(pred3,np.float32)[:,0]; realized=np.asarray(y3,np.float32).reshape(-1,3)[:,0]
    ids=np.flatnonzero((score>0)&np.asarray(eligible,bool)); pick=ids[np.argsort(-score[ids],kind='stable')[:min(int(cap),len(ids))]]
    sel=np.zeros(len(score),bool); sel[pick]=True; ud=np.zeros(len(score),float); ud[pick]=realized[pick]
    return sel,ud,pick,score

def append_fusion_diagnostic(table,fusion):
    """Pre-registered diagnostic only: nested V130 LightGBM fusion c/i/diff; never used by the V131 primary package."""
    fusion=np.asarray(fusion,np.float32); n=len(fusion)
    C=table['cand'].reshape(n,3); I=table['inc'].reshape(n,3); rr=np.arange(n)[:,None]
    fc=np.take_along_axis(fusion,C,1).reshape(-1); fi=np.take_along_axis(fusion,I,1).reshape(-1)
    X=np.c_[table['X'],fc,fi,fc-fi].astype(np.float32)
    names=list(table['features'])+['diag_fusion_c','diag_fusion_i','diag_fusion_diff']
    return X,names

# Fast invariant: primary includes all five BM25 blocks and no pair identity.
_dummy_names=['x','bm25_uni_full_c','bm25_uni_latest_c','bm25_uni_recency_c','bm25_bi_full_c','bm25_bi_latest_c']
assert feature_mask(_dummy_names,'primary').sum()==6 and feature_mask(_dummy_names,'no_bm25').sum()==1
print('V131 global BM25 head ready; primary is fixed all-stream BM25, Ridge alpha',cfg.action_alpha)


action utility helpers ready; hard cap is magnitude-aligned, no benefit:harm purity gate
V131 global BM25 head ready; primary is fixed all-stream BM25, Ridge alpha 10.0


In [9]:

# =========================
# 8. Fully nested V131 world: clean V129/V130 analogue exclusion + global BM25 residual authority
# =========================
def fully_nested_v131_world(indices,world):
    indices=np.asarray(indices,int); local={g:i for i,g in enumerate(indices)}
    wseed=cfg.seed+int(hashlib.sha1(world.encode()).hexdigest()[:6],16)%10000
    outer=make_world_splits(indices,world,wseed); n=len(indices)
    fold_id=np.full(n,-1,int); policy_state=np.full(n,-1,np.int8)
    v129_sel=np.zeros(n,bool); v130_sel=np.zeros(n,bool); eligible=np.zeros(n,bool); selected=np.zeros(n,bool); user_delta=np.zeros(n,float)
    action_pred=np.zeros((n,3),np.float32); action_y=np.zeros((n,3),np.float32); action_c=np.zeros((n,3),np.int16); action_i=np.zeros((n,3),np.int16)
    oof_X=None; feat_names=None; fold_rows=[]
    ablation_ud={k:np.zeros(n,float) for k in ['no_bm25','bm25_full','bm25_temporal','with_lgb_fusion']}

    for f,(tr,va) in enumerate(outer):
        t0=time.time(); bundle=fit_bundle(tr); inner=make_inner_splits(tr,wseed+100+f); pos={g:i for i,g in enumerate(tr)}
        ftr=np.zeros((len(tr),M),np.float32); ranchor_tr=np.zeros_like(ftr); panchor_tr=np.zeros_like(ftr); rtr=None; names=None
        # Inner validation rows receive label-excluded Stage-1/retrieval-anchor and policy-anchor semantics.
        for j,(ir,iv) in enumerate(inner):
            inner_bundle=fit_bundle(ir)
            z=fit_stage1_once(ir,iv,inner_bundle,wseed+10000*f+100*j)
            pa=cached_policy_anchor_split(ir,iv,wseed+20000*f+200*j)
            loc=np.array([pos[g] for g in iv]); ftr[loc]=z['fusion_query']; ranchor_tr[loc]=z['retrieval_anchor_query']; panchor_tr[loc]=pa['val_anchor']
            if rtr is None: rtr=np.zeros((len(tr),M,z['raw_query'].shape[2]),np.float32); names=z['names']
            assert z['names']==names; rtr[loc]=z['raw_query']
            del z,pa,inner_bundle; gc.collect()
        assert np.isfinite(ftr).all() and np.isfinite(ranchor_tr).all() and np.isfinite(panchor_tr).all() and np.isfinite(rtr).all()

        # V129 analogue action head (sparse retrieval-anchor candidate geometry; fixed 5%).
        X129,Y129,*_,fn129=action_table(tr,ranchor_tr,ftr,rtr,names); m129=fit_action_model(X129,Y129)
        # V130 analogue action head (champion-aligned geometry; fixed 2% after V129 analogue exclusion).
        X130,Y130,*_,fn130=action_table(tr,panchor_tr,ftr,rtr,names); m130=fit_action_model(X130,Y130)
        assert fn129==fn130

        # V131 confirmatory global head: target-safe candidates, target-blind raw features, no pair identity.
        gtr=global_action_table(tr,panchor_tr,bundle,tr)
        models={}
        for variant in ['primary','no_bm25','bm25_full','bm25_temporal']:
            models[variant]=fit_global_action_model(gtr['X'],gtr['y'],gtr['features'],variant)
        gtr_lgb_X,gtr_lgb_names=append_fusion_diagnostic(gtr,ftr)
        lgb_diag=make_pipeline(StandardScaler(),Ridge(alpha=cfg.action_alpha,solver='lsqr',tol=1e-5,max_iter=3000)).fit(gtr_lgb_X,gtr['y'])

        # Outer validation is untouched until all three authority layers are trained.
        zv=fit_stage1_once(tr,va,bundle,wseed+50000+f); pav=cached_policy_anchor_split(tr,va,wseed+60000+f)
        a129=predict_action(m129,va,zv['retrieval_anchor_query'],zv['fusion_query'],zv['raw_query'],zv['names'])
        a130=predict_action(m130,va,pav['val_anchor'],zv['fusion_query'],zv['raw_query'],zv['names'])
        gv=global_action_table(va,pav['val_anchor'],bundle,tr)
        assert gv['features']==gtr['features']
        gv_lgb_X,gv_lgb_names=append_fusion_diagnostic(gv,zv['fusion_query']); assert gv_lgb_names==gtr_lgb_names

        lv=np.array([local[g] for g in va]); fold_id[lv]=f; policy_state[lv]=pav['val_state']
        # Clean V129 analogue selection: same executed V129 policy (best rank2-4, 5% of fold population).
        s129,d129,c129,i129,j129=choose_user_actions(a129['pred'],a129['y'],a129['cand'],a129['inc'],len(va))
        cap129=int(math.ceil(cfg.v129_coverage*len(va))); ss129,_,_=select_fixed_cap(s129,d129,c129,(pav['val_state']==0),cap129)
        # Clean V130 analogue selection: state0, disjoint from V129 analogue, fixed 2% of fold population.
        s130,d130,c130,i130,j130=choose_user_actions(a130['pred'],a130['y'],a130['cand'],a130['inc'],len(va))
        cap130=int(math.ceil(cfg.v130_coverage*len(va))); e130=(pav['val_state']==0)&(~ss129); ss130,_,_=select_fixed_cap(s130,d130,c130,e130,cap130)
        ev131=(pav['val_state']==0)&(~ss129)&(~ss130)

        p_primary=predict_global_action(models['primary'][0],models['primary'][1],gv)
        cap131=int(math.ceil(cfg.v131_coverage*len(va))); ss131,ud131,_,_=select_rank2_fixed_cap(p_primary,gv['y'],ev131,cap131)

        v129_sel[lv]=ss129; v130_sel[lv]=ss130; eligible[lv]=ev131; selected[lv]=ss131; user_delta[lv]=ud131
        action_pred[lv]=p_primary; action_y[lv]=gv['y'].reshape(len(va),3); action_c[lv]=gv['cand'].reshape(len(va),3); action_i[lv]=gv['inc'].reshape(len(va),3)
        if oof_X is None:
            oof_X=np.zeros((3*n,gv['X'].shape[1]),np.float32); feat_names=gv['features']
        assert gv['features']==feat_names
        for q,li in enumerate(lv): oof_X[3*li:3*li+3]=gv['X'][3*q:3*q+3]

        fold_ab={}
        for variant in ['no_bm25','bm25_full','bm25_temporal']:
            pv=predict_global_action(models[variant][0],models[variant][1],gv)
            _,uv,_,_=select_rank2_fixed_cap(pv,gv['y'],ev131,cap131); ablation_ud[variant][lv]=uv; fold_ab[variant]=float(uv.mean())
        pv_lgb=lgb_diag.predict(gv_lgb_X).astype(np.float32).reshape(len(va),3)
        _,uv_lgb,_,_=select_rank2_fixed_cap(pv_lgb,gv['y'],ev131,cap131); ablation_ud['with_lgb_fusion'][lv]=uv_lgb; fold_ab['with_lgb_fusion']=float(uv_lgb.mean())
        fold_rows.append({
            'world':world,'fold':f,'n':len(va),'v129_analogue_n':int(ss129.sum()),'v130_analogue_n':int(ss130.sum()),
            'residual_eligible_n':int(ev131.sum()),'v131_n':int(ss131.sum()),'v131_gain':float(ud131.mean()),
            'no_bm25_gain':fold_ab['no_bm25'],'bm25_full_gain':fold_ab['bm25_full'],'bm25_temporal_gain':fold_ab['bm25_temporal'],'with_lgb_fusion_gain':fold_ab['with_lgb_fusion'],
            'policy_anchor_ndcg':ndcg5(y[va],pav['val_anchor']),'fusion_ndcg':ndcg5(y[va],zv['fusion_query']),'s0_rate':float((pav['val_state']==0).mean()),'seconds':time.time()-t0
        })
        del X129,Y129,X130,Y130,m129,m130,gtr,models,lgb_diag,gtr_lgb_X,gv_lgb_X,zv,pav,a129,a130,gv,rtr,bundle; gc.collect()

    assert (fold_id>=0).all() and oof_X is not None and np.isfinite(oof_X).all()
    ciu=bootstrap_ci(user_delta,seed=SEED+41); cig=bootstrap_ci(user_delta,near_group[indices],SEED+42)
    fg=[float(user_delta[fold_id==f].mean()) for f in sorted(np.unique(fold_id))]
    ab={k:{'gain':float(v.mean()),'fold_gains':[float(v[fold_id==f].mean()) for f in sorted(np.unique(fold_id))]} for k,v in ablation_ud.items()}
    return {
        'world':world,'indices':indices,'fold_id':fold_id,'policy_state':policy_state,'v129_selected':v129_sel,'v130_selected':v130_sel,
        'eligible':eligible,'selected':selected,'user_delta':user_delta,'incremental_gain':float(user_delta.mean()),'fold_gains':fg,'ci_user':ciu,'ci_cluster':cig,
        'action_pred':action_pred,'action_y':action_y,'action_cand':action_c,'action_inc':action_i,
        'oof_action_X':oof_X,'oof_action_y':action_y.reshape(-1),'action_feature_names':feat_names,'ablations':ab,'fold_metrics':pd.DataFrame(fold_rows)
    }


In [10]:

# =========================
# 9. PRIMARY random-user V131 confirmatory evaluation
# =========================
t0=time.time(); primary=fully_nested_v131_world(ALL_TRAIN,'random')
primary_pos=int((np.asarray(primary['fold_gains'])>0).sum())
print('PRIMARY V131 incremental gain',primary['incremental_gain'],'actions',int(primary['selected'].sum()),'positive folds',primary_pos,'folds',primary['fold_gains'])
print('CI user',primary['ci_user'],'CI near-cluster',primary['ci_cluster'])
print('Ablations',json.dumps(primary['ablations'],indent=2))
primary['fold_metrics'].to_csv(OUT/'v131_primary_fold_metrics.csv',index=False)
json.dump(primary['action_feature_names'],open(OUT/'v131_action_feature_names.json','w'),indent=2)

# Full primary action opportunity ledger (all 3 policy-anchor challengers per user).
rows=[]; F=primary['oof_action_X'].shape[1]
for li,g in enumerate(primary['indices']):
    for j in range(3):
        rr=3*li+j
        rows.append({
            'user_id':train.user_id.iloc[g],'fold':int(primary['fold_id'][li]),'policy_state':int(primary['policy_state'][li]),
            'v129_analogue_selected':bool(primary['v129_selected'][li]),'v130_analogue_selected':bool(primary['v130_selected'][li]),
            'eligible_v131':bool(primary['eligible'][li]),'selected_v131':bool(primary['selected'][li] and j==0),
            'incumbent':MODULES[int(primary['action_inc'][li,j])],'challenger':MODULES[int(primary['action_cand'][li,j])],
            'candidate_rank':j+2,'pred_utility':float(primary['action_pred'][li,j]),'realized_delta':float(primary['action_y'][li,j])
        })
pd.DataFrame(rows).to_csv(OUT/'v131_action_opportunity_ledger.csv',index=False)
np.savez_compressed(OUT/'v131_primary_oof.npz',action_pred=primary['action_pred'],action_y=primary['action_y'],action_cand=primary['action_cand'],action_inc=primary['action_inc'],
                    fold_id=primary['fold_id'],policy_state=primary['policy_state'],v129_selected=primary['v129_selected'],v130_selected=primary['v130_selected'],
                    eligible=primary['eligible'],selected=primary['selected'],user_delta=primary['user_delta'])
print('primary completed in',round(time.time()-t0,1),'s')


PRIMARY V131 incremental gain 0.00017091690376400947 actions 20 positive folds 3 folds [0.0005707342643290758, -8.037043269723654e-05, 0.0001466307952068746, -2.4606764782220125e-05, 0.0002421966567635536]
CI user (-2.0601319090928883e-05, 0.00039729889018926765) CI near-cluster (-2.1571962822035472e-05, 0.0003577326042507131)
Ablations {
  "no_bm25": {
    "gain": 7.435069046914577e-05,
    "fold_gains": [
      2.1343627013266085e-05,
      -0.00022408293327316642,
      0.00012474986724555492,
      0.00033667281502857804,
      0.00011307007633149625
    ]
  },
  "bm25_full": {
    "gain": 0.00017790576489642263,
    "fold_gains": [
      0.0003856860660016537,
      9.576107608154416e-05,
      0.00040133194997906687,
      -0.0001063203439116478,
      0.00011307007633149625
    ]
  },
  "bm25_temporal": {
    "gain": 4.632090358063579e-05,
    "fold_gains": [
      0.00017575974576175213,
      9.164849296212197e-05,
      -9.512433782219886e-05,
      0.00030124901561066507,
  

In [11]:

# =========================
# 10. Secondary diagnostics: exact/latest/near/contiguous (context; only material collapse vetoes)
# =========================
def world_row(z):
    return {
        'world':z['world'],'incremental_gain':z['incremental_gain'],'actions':int(z['selected'].sum()),
        'positive_folds':int((np.asarray(z['fold_gains'])>0).sum()),'worst_fold':float(np.min(z['fold_gains'])),
        'ci_user_low':z['ci_user'][0],'ci_user_high':z['ci_user'][1],'ci_cluster_low':z['ci_cluster'][0],'ci_cluster_high':z['ci_cluster'][1],
        'no_bm25_gain':z['ablations']['no_bm25']['gain'],'bm25_full_gain':z['ablations']['bm25_full']['gain'],'bm25_temporal_gain':z['ablations']['bm25_temporal']['gain'],'with_lgb_fusion_gain':z['ablations']['with_lgb_fusion']['gain']
    }
world_rows=[world_row(primary)]
if cfg.run_stress:
    for w in ['exact','latest','near','contiguous']:
        t=time.time(); z=fully_nested_v131_world(ALL_TRAIN,w); world_rows.append(world_row(z))
        z['fold_metrics'].to_csv(OUT/f'v131_{w}_fold_metrics.csv',index=False)
        print(w,z['incremental_gain'],'sec',round(time.time()-t,1)); del z; gc.collect()
worlds=pd.DataFrame(world_rows); worlds.to_csv(OUT/'v131_world_metrics.csv',index=False); display(worlds)
severe_stress=bool(cfg.run_stress and (worlds.loc[worlds.world!='random','incremental_gain']<cfg.catastrophic_stress).any())
print('severe_stress',severe_stress,'threshold',cfg.catastrophic_stress)


exact 8.991610910743475e-05 sec 1185.8
latest 6.152329174801708e-05 sec 1187.1
near 0.000256014394108206 sec 1198.1
contiguous 9.423350868746638e-05 sec 1187.3


,world,incremental_gain,actions,positive_folds,worst_fold,ci_user_low,ci_user_high,ci_cluster_low,ci_cluster_high,no_bm25_gain,bm25_full_gain,bm25_temporal_gain,with_lgb_fusion_gain
0,random,0.000171,20,3,-0.000080,-0.000021,0.000397,-0.000022,0.000358,0.000074,0.000178,0.000046,0.000033
1,exact,0.000090,22,2,-0.000254,-0.000145,0.000315,-0.000141,0.000280,0.000004,0.000064,0.000136,0.000008
2,latest,0.000062,23,4,-0.000329,-0.000155,0.000297,-0.000151,0.000268,-0.000107,-0.000041,-0.000101,0.000162
3,near,0.000256,22,5,0.000158,0.000042,0.000497,0.000062,0.000481,0.000126,0.000270,0.000241,0.000391
4,contiguous,0.000094,20,4,-0.000140,-0.000105,0.000286,-0.000087,0.000312,0.000124,0.000170,0.000109,0.000123


severe_stress False threshold -0.0005


In [12]:

# =========================
# 11. Final OOF-trained global BM25 head -> exact V130 + at most five rank-2 Top1 swaps
# =========================
t0=time.time(); test_idx=np.arange(NTR,N,dtype=int)
bundle_all=fit_bundle(ALL_TRAIN)
# Final model uses only clean outer-OOF action rows/labels from the registered random world.
final_model,final_mask=fit_global_action_model(primary['oof_action_X'],primary['oof_action_y'],primary['action_feature_names'],'primary')
test_tab=global_action_table(test_idx,V130_BASE_SCORES,bundle_all,ALL_TRAIN)
assert test_tab['features']==primary['action_feature_names'],'V131 action feature contract drift'
test_pred=predict_global_action(final_model,final_mask,test_tab)
cap=int(math.ceil(cfg.v131_coverage*NTE)); sel,_,pick,score=select_rank2_fixed_cap(test_pred,test_tab['y'],V131_TEST_ELIGIBLE,cap)
assert int(sel.sum())<=5

# Exact extension-only geometry: Top1 <-> current V130 rank2 value swap.
cand_scores=V130_BASE_SCORES.copy(); base_order=np.argsort(-V130_BASE_SCORES,axis=1,kind='stable')
inc=test_tab['inc'].reshape(NTE,3)[:,0].astype(int); cand=test_tab['cand'].reshape(NTE,3)[:,0].astype(int)
for r in np.flatnonzero(sel):
    assert inc[r]==int(base_order[r,0]) and cand[r]==int(base_order[r,1])
    cand_scores[r,inc[r]],cand_scores[r,cand[r]]=cand_scores[r,cand[r]],cand_scores[r,inc[r]]

candidate_path=OUT/'candidate_v131_DIAGNOSTIC.csv'; primary_path=OUT/'submission_v131_primary.csv'; failclose_path=OUT/'submission_v131_failclose_v130.csv'
out=v130_base.copy(); out.loc[:,MODULES]=cand_scores; out.to_csv(candidate_path,index=False); failclose_path.write_bytes(V130_BASE_BYTES)

# Hard correctness gates.
ta,tb=top5_idx(V130_BASE_SCORES),top5_idx(cand_scores); changed=np.any(ta!=tb,axis=1)
membership=np.array([set(a)!=set(b) for a,b in zip(ta,tb)])
immutable_v130=bool(np.array_equal(cand_scores[V130_TEST_SELECTED],V130_BASE_SCORES[V130_TEST_SELECTED]))
locked_noneligible=bool(np.array_equal(cand_scores[~V131_TEST_ELIGIBLE],V130_BASE_SCORES[~V131_TEST_ELIGIBLE]))
rank5_same=bool(np.array_equal(ta[:,4],tb[:,4])); full_a=np.argsort(-V130_BASE_SCORES,axis=1,kind='stable'); full_b=np.argsort(-cand_scores,axis=1,kind='stable')
tail_same=bool(np.array_equal(full_a[:,4:],full_b[:,4:])); multiset_ok=all(np.array_equal(np.sort(V130_BASE_SCORES[r]),np.sort(cand_scores[r])) for r in np.flatnonzero(changed))
schema_ok=bool(out.columns.tolist()==v130_base.columns.tolist() and out.user_id.astype(str).tolist()==test.user_id.astype(str).tolist())
finite_ok=bool(np.isfinite(cand_scores).all()); action_cap_ok=bool(int(changed.sum())==int(sel.sum()) and int(changed.sum())<=5)
rank2_only=bool(all(int(np.flatnonzero(base_order[r]==tb[r,0])[0])+1==2 for r in np.flatnonzero(changed)))
CORRECTNESS_OK=bool(immutable_v130 and locked_noneligible and not membership.any() and rank5_same and tail_same and multiset_ok and schema_ok and finite_ok and action_cap_ok and rank2_only)

# Label-free selected-feature support audit. Pair novelty is descriptive only because pair identity is absent from V131.
fn=primary['action_feature_names']; oof_rank2=primary['oof_action_X'].reshape(NTR,3,-1)[:,0,:]; test_rank2=test_tab['X'].reshape(NTE,3,-1)[:,0,:]
oof_res=primary['eligible']; oof_sel=primary['selected']; test_sel=sel
cand_oof=primary['action_cand'][:,0]; inc_oof=primary['action_inc'][:,0]
cand_counts=Counter(cand_oof[oof_res].tolist()); inc_counts=Counter(inc_oof[oof_res].tolist())
marginal_support_ok=all(cand_counts[int(cand[r])]>0 and inc_counts[int(inc[r])]>0 for r in np.flatnonzero(test_sel))
key_names=['tfidf_word_full_diff','e5_diff','nomic_diff','bm25_uni_full_diff']
key_idx=[fn.index(k) for k in key_names]
lo=np.min(oof_rank2[oof_res][:,key_idx],axis=0); hi=np.max(oof_rank2[oof_res][:,key_idx],axis=0)
unsupported_rows=[]
for r in np.flatnonzero(test_sel):
    z=test_rank2[r,key_idx]; outside=((z<lo)|(z>hi));
    if int(outside.sum())>=3: unsupported_rows.append(int(r))
SUPPORT_FAIL=bool((not marginal_support_ok) or len(unsupported_rows)>0)

def _sel_smd(name):
    k=fn.index(name); return smd(oof_rank2[oof_sel,k],test_rank2[test_sel,k]) if oof_sel.any() and test_sel.any() else 0.0
support_diag={
    'marginal_module_support_ok':bool(marginal_support_ok),'unsupported_numeric_rows':unsupported_rows,
    'pred_utility_smd':smd(primary['action_pred'][:,0][oof_sel],score[test_sel]) if oof_sel.any() and test_sel.any() else 0.0,
    'key_feature_smd':{k:_sel_smd(k) for k in key_names},
    'oof_selected_utility_mean':float(primary['action_pred'][:,0][oof_sel].mean()) if oof_sel.any() else 0.0,
    'test_selected_utility_mean':float(score[test_sel].mean()) if test_sel.any() else 0.0,
}
oof_pairs=np.array([MODULES[int(inc_oof[i])]+'>'+MODULES[int(cand_oof[i])] for i in np.flatnonzero(oof_sel)],object)
test_pairs=np.array([MODULES[int(inc[i])]+'>'+MODULES[int(cand[i])] for i in np.flatnonzero(test_sel)],object)
support_diag['pair_tv_descriptive']=categorical_tv(oof_pairs,test_pairs) if len(test_pairs) else 0.0
support_diag['unseen_pair_rate_descriptive']=float(np.mean([p not in set(oof_pairs.tolist()) for p in test_pairs])) if len(test_pairs) else 0.0

primary_gain=float(primary['incremental_gain']); primary_pos=int((np.asarray(primary['fold_gains'])>0).sum())
if CORRECTNESS_OK and not SUPPORT_FAIL and not severe_stress and primary_gain>=cfg.submit_gain and primary_pos>=4:
    STATUS='SUBMIT'
elif CORRECTNESS_OK and not SUPPORT_FAIL and not severe_stress and primary_gain>=cfg.borderline_gain and primary_pos>=3:
    STATUS='BORDERLINE'
else:
    STATUS='DO NOT SUBMIT'
RECOMMEND_SUBMIT=bool(STATUS=='SUBMIT' and not cfg.smoke)
if RECOMMEND_SUBMIT: primary_path.write_bytes(candidate_path.read_bytes())
else: primary_path.write_bytes(V130_BASE_BYTES)

# Test opportunity ledger: all three actions are logged, but only rank2 is deployable.
opp=[]; X3=test_tab['X'].reshape(NTE,3,-1)
for i,u in enumerate(test.user_id.astype(str)):
    for q in range(3):
        opp.append({
            'user_id':u,'eligible_residual':bool(V131_TEST_ELIGIBLE[i]),'frozen_v130_selected':bool(V130_TEST_SELECTED[i]),
            'deployable_rank2':bool(q==0),'selected_v131':bool(sel[i] and q==0),
            'incumbent':MODULES[int(test_tab['inc'].reshape(NTE,3)[i,q])],'challenger':MODULES[int(test_tab['cand'].reshape(NTE,3)[i,q])],
            'candidate_rank':q+2,'pred_utility':float(test_pred[i,q]),
            'tfidf_word_full_diff':float(X3[i,q,fn.index('tfidf_word_full_diff')]),'e5_diff':float(X3[i,q,fn.index('e5_diff')]),
            'nomic_diff':float(X3[i,q,fn.index('nomic_diff')]),'bm25_uni_full_diff':float(X3[i,q,fn.index('bm25_uni_full_diff')])
        })
pd.DataFrame(opp).to_csv(OUT/'v131_test_action_opportunity_ledger.csv',index=False)

pkg={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'sklearn':__import__('sklearn').__version__,'lightgbm':lgb.__version__,
     'e5_id':E5_ID,'e5_revision':E5_REV,'nomic_id':NOMIC_ID,'nomic_revision':NOMIC_REV,'nomic_code_revision':NOMIC_CODE_REV,
     'bm25_k1':1.2,'bm25_b':0.75,'action_alpha':cfg.action_alpha}
json.dump(pkg,open(OUT/'v131_package_versions.json','w'),indent=2)
report={
    'version':'V131','status':STATUS,'recommend_submit':RECOMMEND_SUBMIT,'correctness_ok':CORRECTNESS_OK,'support_fail':SUPPORT_FAIL,
    'protected_base_public_lb':V130_PUBLIC_LB,'protected_base_sha256':V130_BASE_SHA256,'coverage':cfg.v131_coverage,'rank_family':'rank2_only',
    'primary_incremental_gain':primary_gain,'positive_random_folds':primary_pos,'fold_gains':primary['fold_gains'],'primary_ci_user':primary['ci_user'],'primary_ci_cluster':primary['ci_cluster'],
    'ablations':primary['ablations'],'severe_stress':severe_stress,'catastrophic_stress_threshold':cfg.catastrophic_stress,
    'test_new_actions':int(sel.sum()),'test_total_v130_plus_v131_actions':int(V130_TEST_SELECTED.sum()+sel.sum()),'support_diagnostics':support_diag,
    'immutable_v130_actions':immutable_v130,'locked_noneligible_exact':locked_noneligible,'top5_membership_unchanged':bool(not membership.any()),'rank5_unchanged':rank5_same,'tail_unchanged':tail_same,
    'score_multiset_preserved':multiset_ok,'rank2_only':rank2_only,'candidate_sha256':hashlib.sha256(candidate_path.read_bytes()).hexdigest(),'primary_sha256':hashlib.sha256(primary_path.read_bytes()).hexdigest(),
    'submission_instruction':('SUBMIT submission_v131_primary.csv.' if RECOMMEND_SUBMIT else ('BORDERLINE: primary remains exact V130; candidate_v131_DIAGNOSTIC.csv is preserved for manual slot decision.' if STATUS=='BORDERLINE' else 'DO NOT SUBMIT V131; primary is exact V130 fail-close.'))
}
json.dump(report,open(OUT/'v131_decision_report.json','w'),indent=2)
print(json.dumps(report,indent=2)); print('Outputs',OUT,'elapsed final',round(time.time()-t0,1),'s')


{
  "version": "V131",
  "status": "BORDERLINE",
  "recommend_submit": false,
  "correctness_ok": true,
  "support_fail": false,
  "protected_base_public_lb": 0.66173,
  "protected_base_sha256": "28dd2121c72e910be8a01b31cd43d022c7fdf080d8e628bd8f30abc3a24bc93e",
  "coverage": 0.005,
  "rank_family": "rank2_only",
  "primary_incremental_gain": 0.00017091690376400947,
  "positive_random_folds": 3,
  "fold_gains": [
    0.0005707342643290758,
    -8.037043269723654e-05,
    0.0001466307952068746,
    -2.4606764782220125e-05,
    0.0002421966567635536
  ],
  "primary_ci_user": [
    -2.0601319090928883e-05,
    0.00039729889018926765
  ],
  "primary_ci_cluster": [
    -2.1571962822035472e-05,
    0.0003577326042507131
  ],
  "ablations": {
    "no_bm25": {
      "gain": 7.435069046914577e-05,
      "fold_gains": [
        2.1343627013266085e-05,
        -0.00022408293327316642,
        0.00012474986724555492,
        0.00033667281502857804,
        0.00011307007633149625
      ]
    },
   

## Expected artifacts

The run writes `/kaggle/working/v131/` with the registered validation worlds, fold metrics, BM25 mechanism ablations, OOF/action ledgers, cache/model provenance, the always-preserved `candidate_v131_DIAGNOSTIC.csv`, `submission_v131_failclose_v130.csv`, and `submission_v131_primary.csv`. Read `v131_decision_report.json` first: the primary file is the new candidate only when the confirmatory policy is **SUBMIT**; otherwise it is byte-identical to the protected V130 `0.66173` package.
